# Part 1: Dataset Exploration and Preparation

We explore the RealWaste images, the waste descriptions and the policy documents, then build the pipelines and the train / validation / test splits.

Files needed next to this notebook: the folder `RealWaste` (one sub-folder per class), `waste_descriptions.csv` and `waste_policy_documents.json`. Numbers in the Results come from my runs; a rerun can differ slightly where sampling is involved.

## Step 1: Loading Libraries

In [ ]:
import os
import re
import gc
import json
import random
import hashlib
import pathlib
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# Section A: The RealWaste Images

## Step 2: Images per Class

In [ ]:
data_dir = pathlib.Path("RealWaste")
class_names = sorted([p.name for p in data_dir.glob("*") if p.is_dir()])
print("classes:", len(class_names))
print(class_names)

counts = {}
for name in class_names:
    counts[name] = len(list((data_dir / name).glob("*.jpg")))
img_counts = pd.Series(counts).sort_values(ascending=False)
print(img_counts)

img_counts.plot.barh(title="Images per class")
plt.show()

In [ ]:
print("largest / smallest:", img_counts.max() / img_counts.min())
print((img_counts / img_counts.sum()).round(3))
print("total images:", img_counts.sum())

**Result:** 9 class folders matching the 9 text labels. Plastic (921) and Metal (790) are the largest, Textile Trash (318) the smallest, a ratio of about 2.90 (shares 0.194 down to 0.067). The images are far less balanced than the text, so class weights and a stratified split are needed later.

## Step 3: Image Size and Colour Mode

We open 40 random images per class (360 in total) and record their size and mode.

In [ ]:
rows = []
for name in class_names:
    files = sorted((data_dir / name).glob("*.jpg"))
    for f in random.sample(files, 40):
        with Image.open(f) as img:
            rows.append((name, img.width, img.height, img.mode))

size_table = pd.DataFrame(rows, columns=["class", "width", "height", "mode"])
print(size_table.groupby(["width", "height"]).size())
print(size_table["mode"].value_counts())

**Result:** All 360 sampled images are 524 x 524 RGB. They are square, so resizing to 224 x 224 does not distort them. This is a sample, so uniform size is assumed for the rest.

## Step 4: Look at the Images

Four random images per class.

In [ ]:
fig, axes = plt.subplots(len(class_names), 4, figsize=(10, 2.3 * len(class_names)))
for row, name in enumerate(class_names):
    files = sorted((data_dir / name).glob("*.jpg"))
    for col, f in enumerate(random.sample(files, 4)):
        axes[row, col].imshow(Image.open(f))
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(name, loc="left", fontsize=9)
plt.tight_layout()
plt.show()

**Result:** Top-down photos on a light grey surface, so the background does not give the class away. Lighting varies (some bluish images). Food Organics and Vegetation show speckled piles. Look-alike pairs: Paper and Cardboard, Plastic and Miscellaneous Trash, Food Organics and Vegetation. Miscellaneous Trash has no consistent look. Based on only 36 images.

## Step 5: Brightness and Sharpness

Sharpness is the variance of a simple edge filter: low means blurry. 40 images per class.

In [ ]:
def sharpness(gray):
    g = gray.astype("float32")
    edges = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]
    return float(edges.var())

rows = []
for name in class_names:
    files = sorted((data_dir / name).glob("*.jpg"))
    for f in random.sample(files, 40):
        with Image.open(f) as img:
            gray = np.asarray(img.convert("L").resize((256, 256)))
        rows.append((name, float(gray.mean()), sharpness(gray)))
quality = pd.DataFrame(rows, columns=["class", "brightness", "sharpness"])

In [ ]:
def strip_plot(column, title, xlabel):
    plt.figure(figsize=(8, 5))
    for i, name in enumerate(class_names):
        values = quality[quality["class"] == name][column]
        jitter = np.random.uniform(-0.2, 0.2, len(values))
        plt.scatter(values, i + jitter, s=14, alpha=0.5, color="#2a6f97")
        plt.scatter(values.mean(), i, s=90, color="#e63946", zorder=3)
    plt.yticks(range(len(class_names)), class_names)
    plt.xlabel(xlabel)
    plt.title(title + " (blue = one image, red = class average)")
    plt.tight_layout()
    plt.show()

strip_plot("brightness", "Brightness by class", "Average brightness (0 dark, 255 bright)")
strip_plot("sharpness", "Sharpness by class", "Sharpness score (low = blurry)")

**Result:** Average brightness is similar across classes (about 150 to 162). Vegetation (about 3150) and Food Organics (about 2100) score far sharper than the rest (about 1000); Glass is lowest (about 770). The score responds to texture as well as focus, so the high scores fit their speckled look. A model could use texture as a shortcut.

## Step 6: Exact Duplicate Images

A file fingerprint (MD5) finds byte-identical copies, which could leak between train and test.

In [ ]:
def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hashes = []
for name in class_names:
    for f in (data_dir / name).glob("*.jpg"):
        hashes.append(file_hash(f))

print("images checked:", len(hashes))
print("exact duplicate files:", len(hashes) - len(set(hashes)))

**Result:** 4752 images checked, 0 exact duplicates. Near-duplicates (the same item photographed twice) would not be caught.

# Section B: The Waste Descriptions

## Step 7: Load and Check the Descriptions

In [ ]:
desc = pd.read_csv("waste_descriptions.csv")
print(desc.shape)
desc.info()
print(desc.isna().sum())
desc.head()

**Result:** 5000 rows, 5 text columns. `description` is the input and `category` the target. Only `common_confusion` has missing values (2504).

## Step 8: Duplicates and Conflicting Labels

In [ ]:
print("duplicate rows:", desc.duplicated().sum())
print("duplicate descriptions:", desc["description"].duplicated().sum())
labels_per_description = desc.groupby("description")["category"].nunique()
print("descriptions with more than one category:", (labels_per_description > 1).sum())

**Result:** 7 duplicate rows, 61 duplicate descriptions, 0 conflicting labels. An identical description in train and test would inflate the test score, so duplicates are removed before splitting.

## Step 9: Class Balance

In [ ]:
counts = desc["category"].value_counts()
print(counts)
counts.plot.barh(title="Rows per category")
plt.show()

print("largest / smallest:", counts.max() / counts.min())
print((counts / counts.sum()).round(3))

**Result:** Vegetation is largest (600), Paper smallest (506); shares 0.120 to 0.101, ratio 1.19. A model that always guessed Vegetation would score about 12%. The classes are near-balanced, so no class weights are needed, only a stratified split.

## Step 10: Missing Values in common_confusion

In [ ]:
print(desc.isna().mean())

desc["confusion_missing"] = desc["common_confusion"].isna()
print(desc.groupby("category")["confusion_missing"].mean().round(3))
print(desc.groupby("category")["common_confusion"].nunique())

**Result:** 50.08% of `common_confusion` is missing, spread evenly across categories (0.475 to 0.536). Each category has exactly one distinct note, so the blanks are gaps in a category-level note, not real absences.

## Step 11: Label Leakage

In [ ]:
print(desc.groupby("category")["material_composition"].nunique())
print(desc.groupby("category")["disposal_instruction"].nunique())

**Result:** `material_composition` has 1 value per category and `disposal_instruction` has 5, so with `common_confusion` they contain the answer. Only `description` is used as input; the other columns are kept for the RAG documents.

## Step 12: Description Length

In [ ]:
desc["n_words"] = desc["description"].str.split().str.len()
word_counts = desc["n_words"].value_counts().sort_index()

plt.figure(figsize=(8, 4.5))
bars = plt.bar(word_counts.index, word_counts.values, color="#2a6f97")
plt.bar_label(bars, padding=2, fontsize=9)
plt.xticks(word_counts.index)
plt.xlabel("Words per description")
plt.ylabel("Number of descriptions")
plt.show()

print(desc[desc["n_words"] <= 2].sample(8, random_state=1)[["description", "category"]])

In [ ]:
desc.boxplot(column="n_words", by="category", vert=False, figsize=(8, 5))
plt.suptitle("")
plt.title("Words per description by category")
plt.show()

**Result:** Lengths run from 1 word (21 rows) to 10 (13 rows); most are 4 or 5 words. One- and two-word descriptions ("vegetable waste", "pizza box") give a model very little to go on. Paper is slightly shorter than the others, but the ranges overlap heavily.

## Step 13: Vocabulary and Top Words

In [ ]:
print(desc.sample(10, random_state=1)[["description", "category"]].to_string())

words = desc["description"].str.lower().str.findall(r"[a-z]+")
vocab = collections.Counter()
for word_list in words:
    vocab.update(word_list)
print("vocabulary size:", len(vocab))
print(vocab.most_common(15))

In [ ]:
for category in sorted(desc["category"].unique()):
    category_words = collections.Counter()
    for word_list in words[desc["category"] == category]:
        category_words.update(word_list)
    print(category, "->", [w for w, _ in category_words.most_common(6)])

**Result:** The descriptions are template-built (condition word, size or colour word, item, ending), with a vocabulary of only 309 words. Item words name the category (box, peel, can). Template words (with, sized) appear everywhere. "bottle" is a top word for both Glass and Plastic.

## Step 14: Misleading Material Words

In [ ]:
lower = desc["description"].str.lower()
rows = []
for word in ["glass", "metal", "plastic", "paper", "cardboard"]:
    has_word = lower.str.contains(rf"\b{word}\b")
    same = has_word & (desc["category"].str.lower() == word)
    other = int((has_word & ~same).sum())
    rows.append({"word": word, "rows": int(has_word.sum()), "same class": int(same.sum()),
                 "other class": other, "share other": round(other / has_word.sum(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** A material word often sits in a row of another class ("empty patterned glass old t-shirt" is Textile Trash). Share in another class: cardboard 0.254, plastic 0.239, paper 0.218, metal 0.207, glass 0.101. A keyword model would be misled.

## Step 15: After Removing Duplicates

In [ ]:
desc_unique = desc.drop_duplicates(subset="description")
print(len(desc), len(desc_unique))
print(desc_unique["category"].value_counts(normalize=True).round(3))

**Result:** 5000 rows become 4939 and the class shares barely move (Paper 0.101 to 0.100), so the near-balance holds.

# Section C: The Policy Documents

## Step 16: Load the Policy Documents

In [ ]:
policies = json.load(open("waste_policy_documents.json"))
pol = pd.DataFrame(policies)
pol["n_words"] = pol["document_text"].str.split().str.len()
print(len(policies), "documents")
print(pol[["policy_id", "policy_type", "categories_covered", "n_words"]].to_string(index=False))
print(pol["jurisdiction"].value_counts())

**Result:** 14 documents, all for "Metro City", 86 to 146 words each. Docs 1 to 9 cover one category; docs 10 to 14 cover several.

## Step 17: Document Structure

In [ ]:
header_pattern = re.compile(r"^([A-Za-z][A-Za-z \-/&]+):\s*$", re.M)
for _, row in pol.iterrows():
    print(row["policy_id"], header_pattern.findall(row["document_text"]))

**Result:** Docs 1 to 8 share five headers (Acceptable Items, Non-Acceptable Items, Collection Method, Preparation Instructions, Benefits). Doc 9 (Miscellaneous Trash) has its own layout with no "Acceptable Items". Docs 10 to 14 start with GENERAL REQUIREMENTS and then list each category with no acceptable / non-acceptable split.

## Step 18: Category Coverage

In [ ]:
coverage = collections.Counter()
for categories in pol["categories_covered"]:
    for c in categories:
        coverage[c] += 1
print(pd.Series(coverage).sort_values())

**Result:** Paper appears in 1 document, Food Organics in 2, Glass, Textile Trash, Plastic and Cardboard in 3, Vegetation, Metal and Miscellaneous Trash in 4. Paper has no second source to fall back on.

## Step 19: Repeated Content

In [ ]:
bullets = []
for text in pol["document_text"]:
    for line in text.splitlines():
        if line.startswith("- "):
            bullets.append(line[2:].strip())
repeats = pd.Series(bullets).value_counts()
print(repeats[repeats > 1].head(15))

**Result:** The four GENERAL REQUIREMENTS bullets appear 5 times (once in each of docs 10 to 14), and the Vegetation, Metal and Miscellaneous Trash lists 4 times. The multi-category documents mostly recombine the same lists, so indexing all of them would return near-identical chunks.

## Step 20: Window Glass: Do the Documents Agree?

In [ ]:
def split_sections(text):
    parts = header_pattern.split(text)
    headers = parts[1::2]
    bodies = [b.strip() for b in parts[2::2]]
    return list(zip(headers, bodies))

for _, row in pol.iterrows():
    for header, body in split_sections(row["document_text"]):
        for line in body.splitlines():
            if "window glass" in line.lower():
                print("doc", row["policy_id"], "|", header, "|", line.strip())

**Result:** "Window glass or mirrors" appears in docs 2, 11 and 13. In doc 2 it is under Non-Acceptable Items; in docs 11 and 13 it sits in the GLASS GUIDELINES list, which reads as accepted. The documents disagree, and doc 2 (the category-specific one) agrees with the CSV note.

## Step 21: Policy Language Compared with the Descriptions

In [ ]:
policy_words = set(re.findall(r"[a-z]+", " ".join(pol["document_text"]).lower()))
shared = [w for w in vocab if w in policy_words]
covered = sum(vocab[w] for w in shared) / sum(vocab.values())

print("policy vocabulary:", len(policy_words))
print("description vocabulary:", len(vocab))
print("description words also in the policy text:", len(shared))
print("share of word occurrences covered:", round(covered, 3))

**Result:** Only 69 of the 309 description words appear in the policy text (about 28% of word occurrences, my run). Matching a raw description against policy text would work badly, so the plan is to classify first and then retrieve by category.

# Section D: Pipelines and Splits

## Step 22: Image Pipeline (provided)

The provided code resizes to 224 x 224 and keeps 80% for training and 20% as a hold-out, then cuts the hold-out in half. I removed `.cache()` to save memory.

In [ ]:
BATCH_SIZE = 32
IMG_HEIGHT = 224
IMG_WIDTH = 224

num_classes = len(class_names)
print("Number of classes:", num_classes)
print("Total images found:", len(list(data_dir.glob("*/*.jpg"))))

train_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)

validation_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)

val_batches = tf.data.experimental.cardinality(validation_ds)
test_dataset = validation_ds.take(val_batches // 2)
validation_ds = validation_ds.skip(val_batches // 2)

print("training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("validation batches:", tf.data.experimental.cardinality(validation_ds).numpy())
print("test batches:", tf.data.experimental.cardinality(test_dataset).numpy())

**Result:** 4752 images in 9 classes: 3802 for training and 950 held out, giving 119 training batches and 15 each for validation and test.

## Step 23: Check the Split for Overlap

The provided code cuts the shuffled hold-out with `take` and `skip`. We fingerprint the pixels of validation and test to see whether any image is in both.

In [ ]:
def image_hashes(dataset):
    found = []
    for images, labels in dataset:
        for image in images.numpy():
            found.append(hashlib.md5(image.tobytes()).hexdigest())
    return found

test_hashes = image_hashes(test_dataset)
val_hashes = image_hashes(validation_ds)
print("images read (validation, test):", len(val_hashes), len(test_hashes))
print("images in both:", len(set(test_hashes) & set(val_hashes)))

**Result:** 129 images appear in both validation and test (a first, fuller check found none shared with training). The shuffled hold-out is read twice, once by `take` and once by `skip`, in two different orders. A test score on this split would be inflated.

## Step 24: Fix the Split

Read the 950 hold-out images once, keeping each image together with its own label, then split by position with a stratified split.

In [ ]:
full_holdout = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)

image_list, label_list = [], []
for images, labels in full_holdout:          # one pass: images and labels stay paired
    image_list.append(images.numpy())
    label_list.append(labels.numpy())
all_images = np.concatenate(image_list)
all_labels = np.concatenate(label_list)
label_numbers = all_labels.argmax(axis=1)

val_idx, test_idx = train_test_split(np.arange(len(label_numbers)), test_size=0.5,
                                     stratify=label_numbers, random_state=42)

validation_ds = tf.data.Dataset.from_tensor_slices((all_images[val_idx], all_labels[val_idx])).batch(BATCH_SIZE)
test_dataset = tf.data.Dataset.from_tensor_slices((all_images[test_idx], all_labels[test_idx])).batch(BATCH_SIZE)

print("validation per class:", np.bincount(label_numbers[val_idx]))
print("test per class:      ", np.bincount(label_numbers[test_idx]))

del all_images, all_labels, image_list, label_list, full_holdout
_ = gc.collect()

In [ ]:
test_hashes = image_hashes(test_dataset)
val_hashes = image_hashes(validation_ds)
print("images read (validation, test):", len(val_hashes), len(test_hashes))
print("images in both:", len(set(test_hashes) & set(val_hashes)))

**Result:** The split is now 3802 train, 475 validation and 475 test, with 0 images in both validation and test. Class counts are almost identical in validation and test. My first version read images and labels in two separate passes over the shuffled data, so they were not paired; it showed up in Part 2 as validation accuracy near 0.13. Textile Trash has only 37 and 38 images, so its scores will be noisy.

## Step 25: Text Pipeline and Split

Clean the descriptions, drop duplicates first, turn categories into numbers, then split 80/10/10 with a stratified split.

In [ ]:
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_data = desc[["description", "category"]].copy()
text_data["clean"] = text_data["description"].apply(clean_text)
text_data = text_data.drop_duplicates(subset="clean").reset_index(drop=True)

LABELS = sorted(text_data["category"].unique())
label_to_id = {label: i for i, label in enumerate(LABELS)}
text_data["y"] = text_data["category"].map(label_to_id)

train_data, hold_data = train_test_split(text_data, test_size=0.20, stratify=text_data["y"], random_state=42)
val_data, test_data = train_test_split(hold_data, test_size=0.50, stratify=hold_data["y"], random_state=42)

print("rows after removing duplicates:", len(text_data))
print("train / validation / test:", len(train_data), len(val_data), len(test_data))
print(label_to_id)

# no description may appear in two sets
assert not (set(train_data["clean"]) & set(val_data["clean"]))
assert not (set(train_data["clean"]) & set(test_data["clean"]))
assert not (set(val_data["clean"]) & set(test_data["clean"]))
print("no description appears in more than one set")

**Result:** 4939 rows after removing duplicates; 3951 train, 494 validation, 494 test. Only `description` is used as input. Hyphens are kept and stop-words are not removed, because the descriptions are short and "with" is part of the template.

## Step 26: Documents for the RAG Model

Split each policy document by section and tag every chunk with its category.

In [ ]:
chunks = []
for _, row in pol.iterrows():
    for header, body in split_sections(row["document_text"]):
        categories = row["categories_covered"]
        match = re.match(r"([A-Z ]+) GUIDELINES$", header)
        if len(categories) > 1:
            if header == "GENERAL REQUIREMENTS":
                categories = ["ALL"]
            elif match:
                own = [c for c in categories if c.upper() == match.group(1).strip()]
                categories = own or categories
        chunks.append({"chunk_id": f"policy{row['policy_id']}::{header}", "policy_id": row["policy_id"],
                       "header": header, "category": categories[0],
                       "text": f"{row['policy_type']} | {header}\n{body}"})
chunk_table = pd.DataFrame(chunks)
print("chunks:", len(chunk_table))
print(chunk_table[chunk_table["policy_id"] >= 10][["chunk_id", "category"]].to_string(index=False))

In [ ]:
# the multi-category sections add nothing: every bullet is already in the category's own document
def bullets_of(text):
    return {line[2:].strip() for line in text.splitlines() if line.startswith("- ")}

own_docs = chunk_table[chunk_table["policy_id"] <= 9]
new_bullets = 0
for _, row in chunk_table[(chunk_table["policy_id"] >= 10) & (chunk_table["category"] != "ALL")].iterrows():
    known = set()
    for text in own_docs[own_docs["category"] == row["category"]]["text"]:
        known |= bullets_of(text)
    new_bullets += len(bullets_of(row["text"]) - known)
print("bullets in docs 10-14 not found in the category's own document:", new_bullets)

general = chunk_table[chunk_table["header"] == "GENERAL REQUIREMENTS"]["text"].str.split("\n", n=1).str[1]
print("distinct GENERAL REQUIREMENTS texts:", general.nunique())

In [ ]:
# final corpus: category documents + one general chunk + one disposal card per category
policy_chunks = pd.concat([own_docs, chunk_table[chunk_table["header"] == "GENERAL REQUIREMENTS"].head(1)])

cards = []
for category, group in desc.groupby("category"):
    instructions = sorted(group["disposal_instruction"].dropna().unique())
    note = group["common_confusion"].dropna().unique()
    text = (f"{category} | Disposal card\nComposition: {group['material_composition'].iloc[0]}\n"
            f"Disposal: " + " ".join(instructions))
    if len(note) > 0:
        text += "\nCommon confusion: " + note[0]
    cards.append({"chunk_id": f"card::{category}", "policy_id": None, "header": "Disposal card",
                  "category": category, "text": text})

rag_corpus = pd.concat([policy_chunks, pd.DataFrame(cards)], ignore_index=True)
print("chunks:", len(rag_corpus))
print(rag_corpus["category"].value_counts().to_dict())

**Result:** 67 chunks, of which 54 are kept: 45 policy chunks and 9 disposal cards. No bullet in docs 10 to 14 is missing from the category documents and the 5 general-requirements copies are identical, so the repeated sections are dropped. This also removes the conflicting window-glass placement. Every category has 6 chunks (Miscellaneous Trash 5). Part 4 rebuilds this corpus.

## Summary

- **Images:** 4752 images, 9 classes, uneven (ratio 2.90), 524 x 524 RGB, no exact duplicates. The provided split leaked (129 images in both validation and test); the fixed split is 3802 / 475 / 475, stratified.
- **Descriptions:** 5000 rows (4939 after duplicates), template-built with a 309-word vocabulary. Three columns leak the label, so only `description` is used. Split 3951 / 494 / 494.
- **Policy documents:** 14 documents, uneven coverage, one real contradiction (window glass). The RAG corpus has 54 chunks.
- **Limitations:** size, quality and grid findings come from samples; near-duplicate and mislabelled images were not checked; Textile Trash has few test images; the text is template data, so scores on it flatter a model.

# Part 2: Waste Material Classification with a CNN

We classify waste photos into 9 categories with transfer learning (a frozen MobileNetV2 plus our own layers), then tune it and study its mistakes.

Needs the folder `RealWaste` next to this notebook. Creates the file `waste_cnn.keras`, which Part 5 loads. Numbers in the Results come from my runs; training is not perfectly repeatable, so a rerun can differ by a point or two. Training runs on a CPU: allow about 15 minutes for the main run.

## Step 1: Loading Libraries

In [ ]:
import gc
import random
import hashlib
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

## Step 2: Load the Images

Same pipeline as Part 1: resize to 224 x 224, 80% for training and 20% held out.

In [ ]:
data_dir = pathlib.Path("RealWaste")
class_names = sorted([p.name for p in data_dir.glob("*") if p.is_dir()])
num_classes = len(class_names)

BATCH_SIZE = 32
IMG_HEIGHT = 224
IMG_WIDTH = 224

train_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)

print("classes:", num_classes)
print(class_names)

## Step 3: Validation and Test Sets

The provided code split the hold-out with `take` and `skip` and leaked 129 images between validation and test (see Part 1). Here we read the 950 hold-out images once, keep each image with its own label, and split them by position with a stratified split.

In [ ]:
full_holdout = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)

image_list, label_list = [], []
for images, labels in full_holdout:
    image_list.append(images.numpy())
    label_list.append(labels.numpy())
all_images = np.concatenate(image_list)
all_labels = np.concatenate(label_list)
label_numbers = all_labels.argmax(axis=1)

val_idx, test_idx = train_test_split(np.arange(len(label_numbers)), test_size=0.5,
                                     stratify=label_numbers, random_state=42)

validation_ds = tf.data.Dataset.from_tensor_slices((all_images[val_idx], all_labels[val_idx])).batch(BATCH_SIZE)
test_dataset = tf.data.Dataset.from_tensor_slices((all_images[test_idx], all_labels[test_idx])).batch(BATCH_SIZE)

print("validation per class:", np.bincount(label_numbers[val_idx]))
print("test per class:      ", np.bincount(label_numbers[test_idx]))

In [ ]:
def image_hashes(dataset):
    found = []
    for images, labels in dataset:
        for image in images.numpy():
            found.append(hashlib.md5(image.tobytes()).hexdigest())
    return found

val_hashes = image_hashes(validation_ds)
test_hashes = image_hashes(test_dataset)
print("images (validation, test):", len(val_hashes), len(test_hashes))
print("images in both:", len(set(val_hashes) & set(test_hashes)))

del all_images, all_labels, image_list, label_list, full_holdout
_ = gc.collect()

**Result:** 3802 training, 475 validation and 475 test images, with 0 images in both validation and test. Class counts are almost the same in validation and test; Textile Trash has only 37 and 38.

## Step 4: What the Model Receives

In [ ]:
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

for images, labels in train_ds.take(1):
    print("image batch:", images.shape, images.dtype)
    print("label batch:", labels.shape)
    print("pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

**Result:** No GPU, so training runs on the CPU. Batches are (32, 224, 224, 3) with one-hot labels (32, 9). Pixels run from 0 to 255, so the model rescales them to -1 to 1 itself (what MobileNetV2 was trained on).

## Step 5: Build the Model

MobileNetV2 with ImageNet weights is frozen, so it keeps what it already learned. Our layers on top: pooling, dropout, one 128-unit layer, and 9 outputs.

In [ ]:
base_model = keras.applications.MobileNetV2(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
                                            include_top=False, weights="imagenet")
base_model.trainable = False

model = keras.Sequential([
    keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.Rescaling(1 / 127.5, offset=-1),              # 0..255 -> -1..1
    base_model,
    layers.GlobalAveragePooling2D(name="gap"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

**Result:** 2,423,113 parameters, of which only 165,129 are trained (the two Dense layers); the 2,257,984 in MobileNetV2 stay frozen. The base outputs 7 x 7 x 1280 and pooling turns that into 1280 numbers per image. Freezing first protects the pre-trained features from the random new layers and avoids overfitting on 3802 images.

## Step 6: Class Weights and Training

Errors on small classes count more. Weights come from the training labels only. Early stopping watches validation loss and keeps the best weights.

In [ ]:
train_labels = np.concatenate([labels.numpy().argmax(axis=1) for images, labels in train_ds])
weights = compute_class_weight("balanced", classes=np.arange(num_classes), y=train_labels)
class_weight = {i: w for i, w in enumerate(weights)}
for i, w in class_weight.items():
    print(class_names[i], round(float(w), 2))

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(train_ds, validation_data=validation_ds, epochs=10,
                    class_weight=class_weight, callbacks=[early_stop])

**Result:** Weights run from Textile Trash 1.74 down to Plastic 0.57. Training stopped after epoch 5; the best validation loss (0.5550) was at epoch 2 (validation accuracy about 0.81), so the epoch-2 weights were kept. Training accuracy kept rising (0.76 to 0.89) while validation loss stopped improving: the start of overfitting, which early stopping reacts to.

## Step 7: Evaluate on the Test Set

Accuracy, the confusion matrix and per-class scores. Images and labels are read in the same loop so they stay paired.

In [ ]:
def evaluate_on_test(trained_model, title):
    true_list, pred_list = [], []
    for images, labels in test_dataset:
        true_list.append(labels.numpy().argmax(axis=1))
        pred_list.append(trained_model.predict(images, verbose=0).argmax(axis=1))
    y_true = np.concatenate(true_list)
    y_pred = np.concatenate(pred_list)
    print("test accuracy:", round(float((y_true == y_pred).mean()), 4))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 7))
    plt.imshow(cm, cmap="Blues")
    plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
    plt.yticks(range(num_classes), class_names)
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    print(classification_report(y_true, y_pred, target_names=class_names))
    return y_true, y_pred

y_true, y_pred = evaluate_on_test(model, "Confusion matrix (test set, first model)")

**Result:** In an earlier run this first model scored about 0.79. Strongest: Vegetation (recall 0.97) and Food Organics (0.95). Weakest: Miscellaneous Trash (0.54), a catch-all class with no consistent look. Largest confusions: Plastic as Metal (11) and Metal as Plastic (10), Paper as Cardboard (7), Glass as Plastic (7). Part 1 predicted Paper/Cardboard; Plastic/Metal was a surprise.

## Step 8: Look at the Plastic and Metal Mistakes

Six images of each confusion, with the true and predicted class.

In [ ]:
def find_mistakes(trained_model, true_class, predicted_class, how_many=6):
    found = []
    for images, labels in test_dataset:
        predictions = trained_model.predict(images, verbose=0).argmax(axis=1)
        true_numbers = labels.numpy().argmax(axis=1)
        for image, t, p in zip(images.numpy(), true_numbers, predictions):
            if t == true_class and p == predicted_class:
                found.append(image)
    return found[:how_many]

plastic, metal = class_names.index("Plastic"), class_names.index("Metal")
plt.figure(figsize=(14, 5))
for row, (t, p) in enumerate([(plastic, metal), (metal, plastic)]):
    images = find_mistakes(model, t, p)
    for col, image in enumerate(images):
        plt.subplot(2, 6, row * 6 + col + 1)
        plt.imshow(image.astype("uint8"))
        plt.axis("off")
        plt.title(f"true {class_names[t]}\npred {class_names[p]}", fontsize=8)
plt.tight_layout()
plt.show()

**Result:** Most items carry printed labels or paint that hides the material; some plastic looks metallic (foil pouch, silver cap) and some metal looks like plastic (painted lid, mint tin); several items are crushed. A few are ambiguous even to a person. Based on 12 images.

## Step 9: Tuning the Custom Layers

MobileNetV2 is frozen, so we pass every image through it once, keep the 1280 numbers per image, and train different small heads on those numbers. Compared on validation only.

In [ ]:
extractor = keras.Sequential([keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))] + model.layers[:3])

def get_features(dataset):
    features, labels_list = [], []
    for images, labels in dataset:
        features.append(extractor(images, training=False).numpy())
        labels_list.append(labels.numpy())
    return np.concatenate(features), np.concatenate(labels_list)

X_train, y_train = get_features(train_ds)
X_val, y_val = get_features(validation_ds)
print(X_train.shape, y_train.shape, X_val.shape, y_val.shape)

In [ ]:
def make_head(units=(128,), dropout=0.3, l2=0.0):
    regularizer = keras.regularizers.l2(l2) if l2 else None
    head = keras.Sequential([keras.Input(shape=(1280,)), layers.Dropout(dropout)])
    for i, u in enumerate(units):
        head.add(layers.Dense(u, activation="relu", kernel_regularizer=regularizer))
        if i < len(units) - 1:
            head.add(layers.Dropout(dropout))
    head.add(layers.Dense(num_classes, activation="softmax"))
    return head

def train_head(name, **settings):
    tf.random.set_seed(42)
    head = make_head(**settings)
    head.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    h = head.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=30, batch_size=32,
                 class_weight=class_weight, callbacks=[stop], verbose=0)
    val_loss, val_acc = head.evaluate(X_val, y_val, verbose=0)
    row = {"run": name, "epochs": len(h.history["loss"]), "val_loss": round(val_loss, 4),
           "val_acc": round(val_acc, 4), "train_acc": round(h.history["accuracy"][-1], 4)}
    return head, row

In [ ]:
experiments = {
    "baseline: 128, dropout 0.3": dict(units=(128,), dropout=0.3),
    "more dropout: 128, 0.5": dict(units=(128,), dropout=0.5),
    "wider: 256, dropout 0.3": dict(units=(256,), dropout=0.3),
    "deeper: 256-128, dropout 0.3": dict(units=(256, 128), dropout=0.3),
    "L2 1e-3: 128, dropout 0.3": dict(units=(128,), dropout=0.3, l2=1e-3),
    "combo: 256, dropout 0.5": dict(units=(256,), dropout=0.5),
}

heads, rows = {}, []
for name, settings in experiments.items():
    heads[name], row = train_head(name, **settings)
    rows.append(row)
results = pd.DataFrame(rows)
print(results.to_string(index=False))

**Result:** In my run: baseline 0.6227 validation loss, more dropout 0.5831, wider 0.5801, deeper 0.6269, L2 0.7697, combo (256 units, dropout 0.5) 0.5433. Dropout 0.5 narrowed the train-validation accuracy gap (0.114 to 0.072); L2 did not help; a wider layer helped a little and a deeper one did not. The combo has the lowest validation loss and is chosen. Differences in accuracy are within noise (one validation image is worth 0.2 points), and each run is a single run.

## Step 10: Extra: scikit-learn Models on the Same Features

A second opinion on the same 1280 numbers: logistic regression, SVM and random forest, then cross-validation and a small grid search.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV

train_numbers = y_train.argmax(axis=1)
val_numbers = y_val.argmax(axis=1)

sk_models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "SVM (rbf)": make_pipeline(StandardScaler(), SVC(kernel="rbf", class_weight="balanced")),
    "Random forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
}
rows = []
for name, sk_model in sk_models.items():
    sk_model.fit(X_train, train_numbers)
    pred = sk_model.predict(X_val)
    rows.append({"model": name, "accuracy": round(accuracy_score(val_numbers, pred), 4),
                 "macro F1": round(f1_score(val_numbers, pred, average="macro"), 4)})

best_name = results.loc[results["val_loss"].idxmin(), "run"]
pred = heads[best_name].predict(X_val, verbose=0).argmax(axis=1)
rows.append({"model": "Keras head: " + best_name, "accuracy": round(accuracy_score(val_numbers, pred), 4),
             "macro F1": round(f1_score(val_numbers, pred, average="macro"), 4)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
scores = cross_val_score(logistic, X_train, train_numbers, cv=folds, scoring="accuracy")
print("5-fold accuracy:", scores.round(4), "mean:", round(scores.mean(), 4), "spread:", round(scores.std(), 4))

grid = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
                    param_grid={"logisticregression__C": [0.001, 0.01, 0.1, 1]}, cv=3, scoring="f1_macro")
grid.fit(X_train, train_numbers)
print("best C:", grid.best_params_, "| validation accuracy:", round(accuracy_score(val_numbers, grid.predict(X_val)), 4))

In [ ]:
# do class weights help? train the chosen head with and without them
def head_predictions(use_weights):
    tf.random.set_seed(42)
    head = make_head(**experiments[best_name])
    head.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    head.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=30, batch_size=32,
             class_weight=class_weight if use_weights else None, callbacks=[stop], verbose=0)
    return head.predict(X_val, verbose=0).argmax(axis=1)

weighted = head_predictions(True)
unweighted = head_predictions(False)
table = pd.DataFrame({"with weights": recall_score(val_numbers, weighted, average=None).round(3),
                      "without weights": recall_score(val_numbers, unweighted, average=None).round(3)},
                     index=class_names)
print(table)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42).fit(X_train)       # fitted on training data only
points = pca.transform(X_val)
print("variance kept by 2 components:", pca.explained_variance_ratio_.round(3))

plt.figure(figsize=(8, 6))
for k, name in enumerate(class_names):
    plt.scatter(points[val_numbers == k, 0], points[val_numbers == k, 1], s=14, alpha=0.6, label=name)
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.legend(fontsize=7)
plt.title("Validation features in 2 dimensions")
plt.show()

**Result:** Run this step and note which model has the best validation accuracy, whether cross-validation agrees with the single validation score, whether Plastic recall changes without class weights, and which classes overlap in the plot. The final CNN stays the Keras model, because Part 5 needs one model that takes an image.

## Step 11: Final Model and Test Evaluation

The head with the lowest validation loss is joined to the frozen base and scored once on the test set. The model is saved for Part 5.

In [ ]:
best_name = results.loc[results["val_loss"].idxmin(), "run"]
print("chosen by lowest validation loss:", best_name)

final_model = keras.Sequential([extractor, heads[best_name]])
final_model.compile(loss="categorical_crossentropy", metrics=["accuracy"])

y_true, y_pred = evaluate_on_test(final_model, "Confusion matrix (test set, tuned model)")
final_model.save("waste_cnn.keras")

**Result:** Test accuracy 0.8168 (macro F1 0.82) on 475 images; the same architecture scored 0.84 in an earlier run, so a single run varies by about 2 points. Recall: Vegetation 0.97, Food Organics and Paper 0.93, Glass 0.90, Textile Trash 0.84, Cardboard and Metal 0.78, Plastic 0.76, Miscellaneous Trash 0.57 (weakest). Plastic and Metal are often called Glass, so Glass has the lowest precision (0.69).

## Summary

- **Model:** frozen MobileNetV2 plus dropout 0.5 and a 256-unit layer, trained with class weights and early stopping. Test accuracy 0.8168.
- **Hardest classes:** Miscellaneous Trash and Plastic. The largest confusion is Plastic and Metal (printed or painted surfaces, metallic-looking plastic, crushed items).
- **Limitations:** single runs, small validation and test sets (Textile Trash has about 38 images), CPU training so no large experiments, only 12 mistake images viewed, and only one base model tried.

# Part 3: Waste Description Classification

We classify a short waste description (for example "empty green glass bottle with contents") into one of 9 categories, using scikit-learn. Files needed next to this notebook: `waste_descriptions.csv`.

**Plan:** clean the text, split it, compare simple models, study the mistakes, add word embeddings, test the final model once, and wrap it in a function.

## Step 1: Loading Libraries

Standard packages, then scikit-learn tools, then Keras for the embedding step.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

## Step 2: Load Data

In [ ]:
desc = pd.read_csv("waste_descriptions.csv")
print(desc.shape)
desc.head()

**Result:** 5000 rows and 5 columns. Only `description` (input) and `category` (target) are used. The other three columns are fixed per category, so they would leak the answer (see Part 1).

## Step 3: Clean the Text and Remove Duplicates

We lowercase the text, remove odd characters (keeping hyphens), and drop repeated descriptions **before** splitting so the same text cannot land in two sets.

In [ ]:
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

data = desc[["description", "category"]].copy()
data["clean"] = data["description"].apply(clean_text)

print("rows before:", len(data))
data = data.drop_duplicates(subset="clean").reset_index(drop=True)
print("rows after removing duplicates:", len(data))

**Result:** 5000 rows become 4939.

## Step 4: Labels and Splits

Category names become numbers, then a stratified 80/10/10 split (train, validation, test).

In [ ]:
LABELS = sorted(data["category"].unique())
label_to_id = {label: i for i, label in enumerate(LABELS)}
data["y"] = data["category"].map(label_to_id)

train_data, hold_data = train_test_split(data, test_size=0.20, stratify=data["y"], random_state=42)
val_data, test_data = train_test_split(hold_data, test_size=0.50, stratify=hold_data["y"], random_state=42)

print("train / validation / test:", len(train_data), len(val_data), len(test_data))
print(LABELS)

**Result:** 3951 train, 494 validation, 494 test. The test set stays untouched until Step 13.

## Step 5: Baseline Models

Every model is a pipeline: TF-IDF turns the text into numbers, then a classifier. Scored on the validation set only. The dummy model always guesses the biggest class.

In [ ]:
X_train, y_train = train_data["clean"], train_data["y"]
X_val, y_val = val_data["clean"], val_data["y"]

def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2))

baseline_models = {
    "Dummy (biggest class)": DummyClassifier(strategy="most_frequent"),
    "Naive Bayes": Pipeline([("tfidf", tfidf()), ("clf", MultinomialNB())]),
    "Logistic regression": Pipeline([("tfidf", tfidf()), ("clf", LogisticRegression(max_iter=2000))]),
    "Random forest": Pipeline([("tfidf", tfidf()), ("clf", RandomForestClassifier(n_estimators=300, random_state=42))]),
}

rows = []
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    rows.append({"model": name,
                 "accuracy": round(accuracy_score(y_val, pred), 4),
                 "macro F1": round(f1_score(y_val, pred, average="macro"), 4)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** Dummy 0.119, Naive Bayes 0.990, logistic regression 0.990, random forest 0.972. Near-perfect scores reflect easy template text, not real-world accuracy.

## Step 6: Look at the Mistakes

Print every validation mistake of the two best models.

In [ ]:
def show_mistakes(model, name):
    pred = model.predict(X_val)
    wrong = pred != y_val.values
    table = val_data[wrong][["description", "category"]].copy()
    table["predicted"] = [LABELS[i] for i in pred[wrong]]
    print(name, "-", int(wrong.sum()), "mistakes")
    print(table.to_string())
    print()

show_mistakes(baseline_models["Naive Bayes"], "Naive Bayes")
show_mistakes(baseline_models["Logistic regression"], "Logistic regression")

**Result:** 5 mistakes each, 3 shared. In 9 of the 10 a material word (glass, paper, plastic) sits in front of an item from another class (pen, marker, battery). Miscellaneous Trash is the true class in 8 of 10.

## Step 7: Why Do Material Words Win?

Count how often some words appear in training and which categories they belong to.

In [ ]:
for word in ["pen", "marker", "battery", "toy", "rug", "glass", "paper", "plastic", "cardboard"]:
    has_word = train_data["clean"].str.contains(rf"\b{word}\b")
    counts = train_data[has_word]["category"].value_counts().to_dict()
    print(word, int(has_word.sum()), counts)

**Result:** Item words (pen 19, marker 15, battery 21, rug 22) are rare but always one class. Material words are common (glass 334, paper 255) and mostly point to their own class, so the models follow them. "toy" is split 21 Plastic against 20 Miscellaneous Trash.

## Step 8: Compare Variants with Cross-Validation

Five-fold cross-validation on train and validation together (4445 descriptions) separates models that look the same on 494 validation rows. The test set is not used.

In [ ]:
X_dev = pd.concat([train_data["clean"], val_data["clean"]])
y_dev = pd.concat([train_data["y"], val_data["y"]])
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def tfidf_words():
    return TfidfVectorizer()

variants = {
    "NB, words": Pipeline([("tfidf", tfidf_words()), ("clf", MultinomialNB())]),
    "NB, words+pairs": Pipeline([("tfidf", tfidf()), ("clf", MultinomialNB())]),
    "ComplementNB, words+pairs": Pipeline([("tfidf", tfidf()), ("clf", ComplementNB())]),
    "LR C=1, words": Pipeline([("tfidf", tfidf_words()), ("clf", LogisticRegression(max_iter=2000))]),
    "LR C=1, words+pairs": Pipeline([("tfidf", tfidf()), ("clf", LogisticRegression(max_iter=2000))]),
    "LR C=10, words+pairs": Pipeline([("tfidf", tfidf()), ("clf", LogisticRegression(C=10, max_iter=2000))]),
    "LinearSVC, words+pairs": Pipeline([("tfidf", tfidf()), ("clf", LinearSVC())]),
}

rows = []
for name, model in variants.items():
    scores = cross_val_score(model, X_dev, y_dev, cv=folds, scoring="accuracy")
    rows.append({"variant": name, "mean accuracy": round(scores.mean(), 4), "spread": round(scores.std(), 4)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** LinearSVC is best (0.9998), then logistic regression with C=10 (0.9984) and ComplementNB (0.9966). Raising C from 1 to 10 lifts logistic regression from 0.9928 to 0.9984, so regularisation was suppressing the rare item words. The best of 7 variants is slightly optimistic.

## Step 9: The Two Best on the Validation Set

In [ ]:
best_two = {
    "LinearSVC": Pipeline([("tfidf", tfidf()), ("clf", LinearSVC())]),
    "Logistic regression C=10": Pipeline([("tfidf", tfidf()), ("clf", LogisticRegression(C=10, max_iter=2000))]),
}
for name, model in best_two.items():
    model.fit(X_train, y_train)
    print(name, "validation accuracy:", round(accuracy_score(y_val, model.predict(X_val)), 4))
    show_mistakes(model, name)

**Result:** LinearSVC makes 0 mistakes; logistic regression (C=10) makes 3, all of them decoys ("marker", "pen"). Zero errors on 494 rows is not a zero error rate.

## Step 10: Stress Test on Unusual Descriptions

LinearSVC gives no probabilities, so we wrap it in `CalibratedClassifierCV`. The test inputs are my own, including decoys, natural phrases, and edge cases.

In [ ]:
calibrated = Pipeline([("tfidf", tfidf()), ("clf", CalibratedClassifierCV(LinearSVC(), cv=5))])
calibrated.fit(X_train, y_train)

stress_inputs = [
    ("blue glass pen", "Miscellaneous Trash"), ("green plastic marker", "Miscellaneous Trash"),
    ("dry paper battery", "Miscellaneous Trash"), ("old cardboard rug", "Textile Trash"),
    ("white plastic pen", "Miscellaneous Trash"),
    ("an empty glass jar", "Glass"), ("banana peel", "Food Organics"),
    ("grass clippings from the garden", "Vegetation"), ("cotton t-shirt", "Textile Trash"),
    ("aluminium can", "Metal"), ("used battery", "Miscellaneous Trash"), ("newspaper", "Paper"),
    ("", "?"), ("glass", "Glass"), ("asdf qwerty", "?"), ("chupa ya plastiki", "Plastic"),
]

rows = []
for text, expected in stress_inputs:
    probs = calibrated.predict_proba([clean_text(text)])[0]
    best = probs.argmax()
    rows.append({"input": text, "expected": expected, "predicted": LABELS[best], "confidence": round(probs[best], 2)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** 13 of the 14 answerable inputs are right, including all 5 decoys (the two "pen" decoys at only 0.51). Unknown text (empty, gibberish, Swahili) gets 0.25, below every correct answer, so a confidence threshold can reject it.

## Step 11: Word Embeddings

The brief asks for word embeddings. A small Keras model learns one 32-number vector per word while it learns the categories. These embeddings are learned from the labels, so they are not like Word2Vec.

In [ ]:
max_len = int(train_data["clean"].str.split().str.len().max())

vectorizer = layers.TextVectorization(standardize=None, split="whitespace",
                                      output_mode="int", output_sequence_length=max_len)
vectorizer.adapt(train_data["clean"].values)          # vocabulary from training text only
vocab = vectorizer.get_vocabulary()

X_train_ids = vectorizer(tf.constant(train_data["clean"].values)).numpy()
X_val_ids = vectorizer(tf.constant(val_data["clean"].values)).numpy()

print("longest description (words):", max_len)
print("vocabulary size:", len(vocab))
print("training ids shape:", X_train_ids.shape)

In [ ]:
emb_model = keras.Sequential([
    keras.Input(shape=(max_len,)),
    layers.Embedding(input_dim=len(vocab), output_dim=32, mask_zero=True, name="emb"),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.3),
    layers.Dense(len(LABELS), activation="softmax"),
])
emb_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
history = emb_model.fit(X_train_ids, y_train, validation_data=(X_val_ids, y_val),
                        epochs=100, batch_size=32, callbacks=[early_stop], verbose=0)

val_loss, val_acc = emb_model.evaluate(X_val_ids, y_val, verbose=0)
print("epochs run:", len(history.history["loss"]))
print("validation loss:", round(val_loss, 5), "| validation accuracy:", round(val_acc, 4))

**Result:** Vocabulary of 307 entries, longest description 10 words. After the full 100 epochs validation accuracy is 1.0 (loss 0.00145), tying LinearSVC.

In [ ]:
word_vectors = emb_model.get_layer("emb").get_weights()[0]
words = [str(w) for w in vocab]
unit_vectors = word_vectors / (np.linalg.norm(word_vectors, axis=1, keepdims=True) + 1e-9)

def nearest_words(word, how_many=6):
    i = words.index(word)
    similarity = unit_vectors @ unit_vectors[i]
    order = np.argsort(-similarity)
    result = []
    for j in order:
        if j > 1 and j != i:                      # skip padding, [UNK] and the word itself
            result.append((words[j], round(float(similarity[j]), 2)))
        if len(result) == how_many:
            break
    return result

for word in ["glass", "plastic", "paper", "pen", "marker", "bottle"]:
    print(word, "->", nearest_words(word))

**Result:** Neighbours group by category, not by meaning: glass sits with wine, beer, perfume; plastic with yogurt, milk, jug; pen and marker with other Miscellaneous Trash items (similarity 0.99 to 1.0). Closeness means "the classifier treats them alike".

In [ ]:
def average_embedding(ids):
    mask = (ids > 0)[:, :, None]                  # real words only (0 = padding)
    total = (word_vectors[ids] * mask).sum(axis=1)
    count = np.maximum(mask.sum(axis=1), 1)       # avoid dividing by zero
    return total / count

E_train = average_embedding(X_train_ids)
E_val = average_embedding(X_val_ids)
print(E_train.shape, E_val.shape)

embedding_models = {
    "Embeddings + logistic regression": LogisticRegression(max_iter=2000),
    "Embeddings + LinearSVC": LinearSVC(),
    "Embeddings + random forest": RandomForestClassifier(n_estimators=300, random_state=42),
}
rows = []
for name, model in embedding_models.items():
    model.fit(E_train, y_train)
    pred = model.predict(E_val)
    rows.append({"model": name, "accuracy": round(accuracy_score(y_val, pred), 4)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** The 32-number averages give 1.000, 1.000 and 0.998, matching TF-IDF with far fewer numbers. The embeddings were learned from the labels, so this is not independent proof.

In [ ]:
empty_ids = vectorizer(tf.constant([""])).numpy()
print(emb_model.predict(empty_ids, verbose=0))

**Result:** An empty description gives `nan` instead of probabilities (every position is padding, so the average divides by zero). The final prediction function therefore checks for empty and unknown input itself.

## Step 12: Final Model and One Test Evaluation

The final choice was made from cross-validation before touching the test set: TF-IDF, word pairs, calibrated LinearSVC, refit on train and validation together.

In [ ]:
text_model = Pipeline([("tfidf", tfidf()), ("clf", CalibratedClassifierCV(LinearSVC(), cv=5))])
text_model.fit(X_dev, y_dev)

X_test, y_test = test_data["clean"], test_data["y"]
test_pred = text_model.predict(X_test)
print("test accuracy:", round(accuracy_score(y_test, test_pred), 4))

cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(len(LABELS)), LABELS, rotation=45, ha="right")
plt.yticks(range(len(LABELS)), LABELS)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

print(classification_report(y_test, test_pred, target_names=LABELS))

**Result:** Test accuracy 1.0 with 0 errors on 494 descriptions, every precision and recall 1.00. This reflects the template-generated data, not real-world performance; the mistakes studied in Steps 6 and 9 and the stress test are the real analysis.

## Step 13: Prediction Function

A function that returns the category, a confidence and a status, and refuses empty or unknown input instead of guessing. The 0.40 threshold lies between the 0.25 given to unknown text and the lowest correct confidence.

In [ ]:
CONFIDENCE_THRESHOLD = 0.40

def predict_waste_category(text, threshold=CONFIDENCE_THRESHOLD):
    if not isinstance(text, str) or clean_text(text) == "":
        return {"category": None, "confidence": 0.0, "status": "empty input", "top3": []}

    cleaned = clean_text(text)
    word_numbers = text_model.named_steps["tfidf"].transform([cleaned])
    if word_numbers.nnz == 0:
        return {"category": None, "confidence": 0.0, "status": "no known words", "top3": []}

    probs = text_model.predict_proba([cleaned])[0]
    order = np.argsort(-probs)[:3]
    top3 = [(LABELS[i], round(float(probs[i]), 2)) for i in order]
    status = "ok" if probs[order[0]] >= threshold else "low confidence"
    return {"category": LABELS[order[0]], "confidence": round(float(probs[order[0]]), 2),
            "status": status, "top3": top3}

for text in ["blue glass pen", "banana peel", "Dry GLASS bottle with contents", "",
             "   ", None, 123, "!!!", "asdf qwerty", "chupa ya plastiki"]:
    print(repr(text), "->", predict_waste_category(text))

**Result:** Four inputs get a category; empty, whitespace, `None`, a number, punctuation only, and text with no known words all return `None` with a reason. With the final model the lowest correct confidence is 0.52, only 0.12 above the threshold, so the threshold should be re-checked whenever the model changes.

## Summary

- **Final model:** TF-IDF (words and word pairs) with a calibrated LinearSVC. Test accuracy 1.0, 0 errors.
- **What the mistakes taught:** a frequent material word (glass, paper) outweighs a rare item word (pen, marker). Loosening the regularisation fixed it.
- **Limitations:** the descriptions are template-generated (307-word vocabulary), so near-perfect accuracy reflects the data. The model cannot read words it never saw, including other languages. The stress test has only 16 hand-written inputs, and the 0.40 threshold is tentative.

# Part 4: Recycling Instruction Generation with RAG

Given a waste category, the system finds the matching policy text, a fine-tuned small language model (`flan-t5-small`) writes the recycling instructions from it, and a checker verifies the answer against that text.

Needs `waste_descriptions.csv` and `waste_policy_documents.json` next to this notebook, plus PyTorch, transformers and sentence-transformers (`pip install torch --index-url https://download.pytorch.org/whl/cpu`, then `pip install transformers sentence-transformers textstat`). Creates the folder `waste_rag_model`, which Part 5 loads. Fine-tuning takes about 20 minutes on a CPU (the unseen-category run about 17). Numbers in the Results come from my runs.

## Step 1: Loading Libraries and Checking the Machine

In [ ]:
import os
os.environ["USE_TF"] = "0"          # keep transformers from loading TensorFlow
import re
import json
import time
import copy
import random
import collections
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
print("torch", torch.__version__)

**Result:** Python 3.13 on Windows 11, CPU only, 16.9 GB of RAM (about 10 GB free after closing other notebooks; only 2.2 GB before, which is the real limit for loading models). torch 2.14.1+cpu, transformers 5.18.0, sentence-transformers 6.1.0.

## Step 2: Policy Chunks

Split every policy document by section and tag each chunk with its category.

In [ ]:
desc = pd.read_csv("waste_descriptions.csv")
policies = json.load(open("waste_policy_documents.json"))
header_pattern = re.compile(r"^([A-Za-z][A-Za-z \-/&]+):\s*$", re.M)

def split_sections(text):
    parts = header_pattern.split(text)
    return list(zip(parts[1::2], [b.strip() for b in parts[2::2]]))

chunks = []
for policy in policies:
    for header, body in split_sections(policy["document_text"]):
        categories = policy["categories_covered"]
        match = re.match(r"([A-Z ]+) GUIDELINES$", header)
        if len(categories) > 1:
            if header == "GENERAL REQUIREMENTS":
                categories = ["ALL"]
            elif match:
                own = [c for c in categories if c.upper() == match.group(1).strip()]
                categories = own or categories
        chunks.append({"chunk_id": f"policy{policy['policy_id']}::{header}", "policy_id": policy["policy_id"],
                       "header": header, "category": categories[0],
                       "text": f"{policy['policy_type']} | {header}\n{body}"})
chunk_table = pd.DataFrame(chunks)
print("chunks before filtering:", len(chunk_table))

In [ ]:
# keep the category documents (1-9) and ONE copy of the general rules
own_docs = chunk_table[chunk_table["policy_id"] <= 9]
general_chunk = chunk_table[chunk_table["header"] == "GENERAL REQUIREMENTS"].head(1)

# one disposal card per category, built from the CSV
cards = []
for category, group in desc.groupby("category"):
    instructions = sorted(group["disposal_instruction"].dropna().unique())
    note = group["common_confusion"].dropna().unique()
    text = (f"{category} | Disposal card\nComposition: {group['material_composition'].iloc[0]}\n"
            f"Disposal: " + " ".join(instructions))
    if len(note) > 0:
        text += "\nCommon confusion: " + note[0]
    cards.append({"chunk_id": f"card::{category}", "policy_id": None, "header": "Disposal card",
                  "category": category, "text": text})

rag_corpus = pd.concat([own_docs, general_chunk, pd.DataFrame(cards)], ignore_index=True)
CATEGORIES = sorted(desc["category"].unique())

print("chunks:", len(rag_corpus))
print(rag_corpus["category"].value_counts().to_dict())
print(rag_corpus["text"].str.split().str.len().describe().round(0).to_dict())

**Result:** 67 chunks, 54 kept: 45 policy chunks and 9 disposal cards. Docs 10 to 14 are dropped because every bullet in them is already in the category's own document, and keeping them would risk contradictory placements (window glass). 6 chunks per category (Miscellaneous Trash 5, its document has four sections) plus one general chunk. Chunks are short (median 26 words, maximum 75), so no further splitting is needed.

## Step 3: Embeddings and the Retrieval Index

Two indexes over the 54 chunks: TF-IDF (word matching) and dense MiniLM embeddings (384 numbers per chunk, meaning matching). For 54 chunks an index is just a matrix plus a cosine search.

In [ ]:
chunk_texts = rag_corpus["text"].tolist()
chunk_category = rag_corpus["category"].values

tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
tfidf_matrix = tfidf.fit_transform(chunk_texts)

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
dense_matrix = encoder.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=False)

print("TF-IDF index:", tfidf_matrix.shape)
print("dense index:", dense_matrix.shape)

In [ ]:
def tfidf_scores(question):
    return cosine_similarity(tfidf.transform([question]), tfidf_matrix)[0]

def dense_scores(question):
    return dense_matrix @ encoder.encode([question], normalize_embeddings=True)[0]

def hybrid_scores(question, alpha=0.5):
    a, b = tfidf_scores(question), dense_scores(question)
    a = (a - a.mean()) / (a.std() + 1e-9)
    b = (b - b.mean()) / (b.std() + 1e-9)
    return alpha * a + (1 - alpha) * b           # alpha is the weight of TF-IDF

## Step 4: Compare the Retrieval Methods

A chunk counts as relevant if it belongs to the question's category. 36 category questions ("How to recycle Glass", 4 wordings x 9 categories) and 18 item questions without the category name ("How do I recycle a pizza box?", written by me).

In [ ]:
wordings = ["How to recycle {c}", "How do I dispose of {c} properly?",
            "What is the correct way to recycle {c}?", "{c} recycling rules"]
category_questions = [(w.format(c=c), c) for c in CATEGORIES for w in wordings]

item_questions = [
    ("How do I recycle a pizza box?", "Cardboard"), ("What should I do with shipping boxes?", "Cardboard"),
    ("How do I dispose of banana peels?", "Food Organics"), ("Can I compost coffee grounds and eggshells?", "Food Organics"),
    ("How do I recycle a wine bottle?", "Glass"), ("What do I do with a broken glass jar?", "Glass"),
    ("How do I recycle aluminium cans?", "Metal"), ("Can I recycle an empty aerosol can?", "Metal"),
    ("How do I dispose of old batteries?", "Miscellaneous Trash"), ("How do I dispose of light bulbs?", "Miscellaneous Trash"),
    ("How do I recycle old newspapers?", "Paper"), ("What do I do with shredded documents?", "Paper"),
    ("How do I recycle a plastic bottle?", "Plastic"), ("Can I recycle plastic bags?", "Plastic"),
    ("What should I do with old clothes?", "Textile Trash"), ("How do I dispose of worn towels?", "Textile Trash"),
    ("What do I do with grass clippings?", "Vegetation"), ("How do I dispose of tree branches?", "Vegetation"),
]

In [ ]:
def evaluate_retrieval(scores_function, questions):
    hit1, hit3, hit5, reciprocal = [], [], [], []
    for question, category in questions:
        order = np.argsort(-scores_function(question))
        relevant = chunk_category[order] == category
        hit1.append(relevant[:1].any())
        hit3.append(relevant[:3].any())
        hit5.append(relevant[:5].any())
        first = np.where(relevant)[0]
        reciprocal.append(1 / (first[0] + 1) if len(first) else 0)
    return {"MRR": round(float(np.mean(reciprocal)), 3), "hit@1": round(float(np.mean(hit1)), 3),
            "hit@3": round(float(np.mean(hit3)), 3), "hit@5": round(float(np.mean(hit5)), 3)}

rows = []
methods = {"TF-IDF": tfidf_scores, "dense (MiniLM)": dense_scores, "hybrid 0.5": hybrid_scores}
for method_name, scores_function in methods.items():
    for set_name, questions in [("category questions", category_questions), ("item questions", item_questions)]:
        rows.append({"method": method_name, "questions": set_name, **evaluate_retrieval(scores_function, questions)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** Category questions: TF-IDF and the hybrid are best (35 of 36 right at rank 1), because the question shares the category name with the chunk. Item questions: dense is best (16 of 18 at rank 1, MRR 0.944, against 12 of 18 and 0.783 for TF-IDF), because linking an item to a category needs meaning. The set is small (one item question is worth 5.6 points) and "relevant" only checks the category.

## Step 5: Choose the Hybrid Weight and Look at the Failures

In [ ]:
rows = []
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    scores_function = lambda question, a=alpha: hybrid_scores(question, a)
    cat_result = evaluate_retrieval(scores_function, category_questions)
    item_result = evaluate_retrieval(scores_function, item_questions)
    rows.append({"alpha (TF-IDF weight)": alpha, "category MRR": cat_result["MRR"],
                 "category hit@1": cat_result["hit@1"], "item MRR": item_result["MRR"], "item hit@1": item_result["hit@1"]})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
def show_failures(scores_function, questions, name):
    print(name, "- questions whose top result is the wrong category")
    for question, category in questions:
        order = np.argsort(-scores_function(question))
        if chunk_category[order[0]] != category:
            top_three = [rag_corpus["chunk_id"][j] for j in order[:3]]
            print("  ", question, "| expected", category, "->", top_three)

show_failures(tfidf_scores, item_questions, "TF-IDF")
show_failures(dense_scores, item_questions, "dense")

**Result:** alpha = 0.25 is best on both sets: MRR 1.000 on category questions (36 of 36) and 0.944 on item questions (16 of 18), as good as pure dense. It was tuned on the same questions it is scored on, so it is slightly optimistic. TF-IDF fails when generic words ("recycle") match the generic Preparation chunks of Glass and Plastic. "banana peels" fails for both: the word "banana" is in no document (they say "fruit and vegetable scraps"), which is why an item should first be classified into a category. "light bulbs" returns the Glass Non-Acceptable chunk first, which does list light bulbs, so that result is relevant even though the category differs. Decision: hybrid with alpha 0.25, restricted to the requested category when it is known.

## Step 6: Reference Answers

The model needs a target answer per category. We build it mechanically from the sources (collection method, preparation steps, items not to recycle, the CSV disposal sentences, and the confusion note), so nothing is written from memory. Miscellaneous Trash has its own layout.

In [ ]:
def chunk_text(category, header):
    row = rag_corpus[(rag_corpus["category"] == category) & (rag_corpus["header"] == header)]
    if row.empty:
        return ""
    return row["text"].iloc[0].split("\n", 1)[1].strip()

def bullets_of(body):
    return [line[2:].strip() for line in body.splitlines() if line.startswith("- ")]

def prose_of(body):
    lines = [line.strip() for line in body.splitlines() if line.strip() and not line.startswith("- ")]
    return " ".join(lines)

In [ ]:
def reference_answer(category):
    card = chunk_text(category, "Disposal card")
    disposal = card.split("Disposal: ")[1].split("\nCommon confusion:")[0].strip()
    note = ""
    if "Common confusion: " in card:
        note = card.split("Common confusion: ")[1].strip()

    if category == "Miscellaneous Trash":
        cannot = bullets_of(chunk_text(category, "Items That Cannot Be Recycled"))
        special = bullets_of(chunk_text(category, "Special Handling Items"))
        text = (f"{category} cannot be recycled as ordinary material. Cannot be recycled: {'; '.join(cannot)}. "
                f"Special handling: {'; '.join(special)}. {disposal}")
    else:
        collect = prose_of(chunk_text(category, "Collection Method"))
        prepare = bullets_of(chunk_text(category, "Preparation Instructions"))
        avoid = bullets_of(chunk_text(category, "Non-Acceptable Items"))
        text = (f"To recycle {category}: {collect} Before recycling: {'; '.join(prepare)}. "
                f"Do not recycle: {'; '.join(avoid)}. {disposal}")
    if note:
        text += " Note: " + note
    return text.strip()

references = {c: reference_answer(c) for c in CATEGORIES}
print(references["Glass"])
print({c: len(r.split()) for c, r in references.items()})

**Result:** Each reference is 95 to 110 words and every sentence comes from the Glass (or other category's) policy sections or the CSV card. They repeat themselves a little ("Rinse containers" then "Rinse container before recycling") because the policy text and the CSV overlap, which lowers readability. Each reference is built from the same text retrieval returns, so a model could pass by copying; the unseen-category test (Step 18) checks for that.

## Step 7: Training Examples

Input: "Answer the question using only the context. Question: ... Context: ...". Target: the category's reference answer. The context always holds the chunks the reference needs plus 0 to 2 chunks from other categories, shuffled, so the model must pick the right ones. If the needed chunks were sometimes missing, the model would learn to invent.

In [ ]:
os.environ["USE_TF"] = "0"
tok = AutoTokenizer.from_pretrained("google/flan-t5-small")

needed_headers = ["Collection Method", "Preparation Instructions", "Non-Acceptable Items", "Disposal card"]
misc_headers = ["Items That Cannot Be Recycled", "Special Handling Items", "Disposal card"]

def needed_chunks(category):
    headers = misc_headers if category == "Miscellaneous Trash" else needed_headers
    found = []
    for header in headers:
        match = rag_corpus[(rag_corpus["category"] == category) & (rag_corpus["header"] == header)]
        found.append(match.iloc[0])
    return found

def make_input(question, chunk_rows):
    context = " ".join(" ".join(row["text"].split()) for row in chunk_rows)
    return f"Answer the question using only the context. Question: {question} Context: {context}"

In [ ]:
train_wordings = ["How to recycle {c}", "How do I dispose of {c} properly?", "What is the correct way to recycle {c}?",
                  "{c} recycling rules", "Explain how to recycle {c}", "What should I do with {c}?",
                  "Give me instructions for {c}", "How can I get rid of {c} responsibly?"]
val_wordings = ["Tell me how to recycle {c}", "Instructions for recycling {c}"]

def build_examples(categories, wordings, copies):
    examples = []
    for category in categories:
        needed = needed_chunks(category)
        others = rag_corpus[(rag_corpus["category"] != category) & (rag_corpus["category"] != "ALL")]
        for wording in wordings:
            for n_distractors in (0, 1, 2):
                for _ in range(copies):
                    picked = [others.iloc[i] for i in random.sample(range(len(others)), n_distractors)]
                    chunk_rows = needed + picked
                    random.shuffle(chunk_rows)
                    question = wording.format(c=category)
                    examples.append({"category": category, "question": question,
                                     "input": make_input(question, chunk_rows), "target": references[category]})
    return pd.DataFrame(examples)

train_examples = build_examples(CATEGORIES, train_wordings, copies=2)
val_examples = build_examples(CATEGORIES, val_wordings, copies=1)
print("examples (train, validation):", len(train_examples), len(val_examples))

input_lengths = np.array([len(tok(s).input_ids) for s in train_examples["input"]])
target_lengths = np.array([len(tok(s).input_ids) for s in train_examples["target"]])
print("input tokens: mean", round(input_lengths.mean()), "max", input_lengths.max(), "| over 512:", int((input_lengths > 512).sum()))
print("target tokens: mean", round(target_lengths.mean()), "max", target_lengths.max())

**Result:** 432 training and 54 validation examples (8 training wordings, 2 different validation wordings). Inputs are 273 tokens on average, 462 at most, none over the 512 limit; targets are 152 tokens on average, 170 at most.

## Step 8: The Model Before Fine-Tuning

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").float()
print("parameters:", sum(p.numel() for p in model.parameters()))

example = train_examples[train_examples["category"] == "Glass"].iloc[0]
inputs = tok(example["input"], return_tensors="pt", truncation=True, max_length=512)
model.eval()
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=200, num_beams=1)
print("QUESTION:", example["question"])
print("UNTRAINED ANSWER:", tok.decode(output[0], skip_special_tokens=True))

**Result:** `flan-t5-small` has 76,961,152 parameters. The untrained model answers only "Glass recycling guidelines | Non-Acceptable Items", a heading copied from the context with no instructions. It does not know the answer format, so fine-tuning is needed.

## Step 9: Fine-Tune the Model

A plain PyTorch loop. Padding is excluded from the loss, gradients are clipped, and the weights with the best validation loss are kept (like `EarlyStopping`).

In [ ]:
def encode_batch(inputs, targets):
    batch = tok(inputs, max_length=512, truncation=True, padding=True, return_tensors="pt")
    labels = tok(text_target=targets, max_length=200, truncation=True, padding=True, return_tensors="pt").input_ids
    labels[labels == tok.pad_token_id] = -100          # padding must not count in the loss
    batch["labels"] = labels
    return batch

def make_loader(examples, batch_size, shuffle):
    pairs = list(zip(examples["input"], examples["target"]))
    return DataLoader(pairs, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=lambda b: encode_batch([x[0] for x in b], [x[1] for x in b]))

def average_loss(model, loader):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            size = batch["input_ids"].size(0)
            total += model(**batch).loss.item() * size
            count += size
    return total / count

In [ ]:
def finetune(train_set, val_set, epochs=3, batch_size=8, lr=3e-4):
    torch.manual_seed(42)
    new_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").float()
    optimizer = torch.optim.AdamW(new_model.parameters(), lr=lr)
    train_loader = make_loader(train_set, batch_size, True)
    val_loader = make_loader(val_set, batch_size, False)
    best_loss, best_weights, history = float("inf"), None, []

    for epoch in range(1, epochs + 1):
        new_model.train()
        start, running = time.time(), 0.0
        for step_number, batch in enumerate(train_loader, 1):
            loss = new_model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(new_model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            running += loss.item()
            if step_number % 10 == 0:
                print(f"  epoch {epoch} step {step_number}/{len(train_loader)} loss {running / step_number:.4f}", flush=True)
        val_loss = average_loss(new_model, val_loader)
        history.append({"epoch": epoch, "train_loss": round(running / len(train_loader), 4),
                        "val_loss": round(val_loss, 4), "seconds": round(time.time() - start)})
        print(history[-1], flush=True)
        if val_loss < best_loss:
            best_loss, best_weights = val_loss, copy.deepcopy(new_model.state_dict())
    new_model.load_state_dict(best_weights)
    return new_model, pd.DataFrame(history)

In [ ]:
# 24 examples per category (216 in total) keeps the run to about 20 minutes on a CPU
train_small = pd.concat([g.sample(24, random_state=42) for _, g in train_examples.groupby("category")])
print("training examples:", len(train_small), "| validation examples:", len(val_examples))

start = time.time()
final_model, history_table = finetune(train_small, val_examples, epochs=3)
print(history_table.to_string(index=False))
print("total minutes:", round((time.time() - start) / 60, 1))

final_model.save_pretrained("waste_rag_model")
tok.save_pretrained("waste_rag_model")

**Result:** 3 epochs in 20.2 minutes (about 6.8 per epoch). Training loss 0.2996, 0.0347, 0.0185; validation loss 0.0264, 0.0084, 0.0098, so the epoch-2 weights were kept. Losses are very low because the task is mostly restructuring text that sits in the context. Validation uses the same nine categories as training, so it shows robustness to new wordings, not to an unseen category.

## Step 10: The Fine-Tuned Answer for Glass

In [ ]:
final_model.eval()
example = train_examples[train_examples["category"] == "Glass"].iloc[0]
inputs = tok(example["input"], return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    output = final_model.generate(**inputs, max_new_tokens=200, num_beams=1)
answer = tok.decode(output[0], skip_special_tokens=True)
print("FINE-TUNED:", answer)
print("same as the reference:", answer.strip() == example["target"].strip())

**Result:** The model now writes the complete Glass answer, identical to the reference (95 words). Glass was a training category, so this alone does not show it reads the context. The next steps test that.

## Step 11: Does It Read the Context? (Glass)

Two tests. A: change one fact in the context (the collection sentence) to something the model never saw. B: remove the Non-Acceptable Items chunk.

In [ ]:
def generate_answer(trained_model, question, chunk_rows, **options):
    inputs = tok(make_input(question, chunk_rows), return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output = trained_model.generate(**inputs, max_new_tokens=200, **options)
    return tok.decode(output[0], skip_special_tokens=True)

glass_chunks = needed_chunks("Glass")

# Test A: edit the collection sentence
edited = []
for row in glass_chunks:
    row = row.copy()
    if row["header"] == "Collection Method":
        row["text"] = row["text"].replace("Place in dedicated glass recycling bins.",
                                          "Take glass to the municipal depot on Saturdays.")
    edited.append(row)
answer_a = generate_answer(final_model, "How to recycle Glass", edited)
print("A:", answer_a[:160])
print("new sentence present:", "municipal depot" in answer_a, "| old sentence present:", "dedicated glass recycling bins" in answer_a)

# Test B: remove the Non-Acceptable Items chunk
without = [row for row in glass_chunks if row["header"] != "Non-Acceptable Items"]
answer_b = generate_answer(final_model, "How to recycle Glass", without)
print("\nB:", answer_b)
print("still lists 'Window glass or mirrors':", "Window glass or mirrors" in answer_b)

**Result:** A: the model wrote the new sentence and dropped the old one, so it copies from the context. B: it did not invent a replacement list, but it still wrote the heading "Do not recycle:" with nothing under it, so the next sentence ("Broken glass should be wrapped...") reads as the item not to recycle. It has memorised the layout of the answer.

## Step 12: The Same Two Tests for All Nine Categories

In [ ]:
removed_header = {"Miscellaneous Trash": "Items That Cannot Be Recycled"}
section_label = {"Miscellaneous Trash": "Cannot be recycled:"}

def first_sentence(text):
    sentence = text.split(". ")[0].strip()
    return sentence if sentence.endswith(".") else sentence + "."

rows = []
for category in CATEGORIES:
    question = f"How to recycle {category}"
    needed = needed_chunks(category)

    # Test A: change one fact
    edited, old_fact, new_fact = [], None, None
    for row in needed:
        row = row.copy()
        if category == "Miscellaneous Trash" and row["header"] == "Special Handling Items":
            old_fact = "Batteries (take to dedicated collection points)"
            new_fact = "Batteries (take to the town hall on Mondays)"
            row["text"] = row["text"].replace(old_fact, new_fact)
        elif category != "Miscellaneous Trash" and row["header"] == "Collection Method":
            old_fact = first_sentence(prose_of(chunk_text(category, "Collection Method")))
            new_fact = f"Take {category.lower()} to the municipal depot on Saturdays."
            row["text"] = row["text"].replace(old_fact, new_fact)
        edited.append(row)
    answer_a = generate_answer(final_model, question, edited)

    # Test B: remove the "do not recycle" section
    header = removed_header.get(category, "Non-Acceptable Items")
    items = bullets_of(chunk_text(category, header))
    answer_b = generate_answer(final_model, question, [r for r in needed if r["header"] != header])
    label = section_label.get(category, "Do not recycle:")
    after = answer_b.split(label)[1].strip()[:45] if label in answer_b else "(section missing)"

    rows.append({"category": category, "new fact used": new_fact in answer_a, "old fact gone": old_fact not in answer_a,
                 "invented items": sum(item in answer_b for item in items), "text after heading": after})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** A: 9 of 9 categories used the edited fact and dropped the old one. B: 0 invented items in all 9. But in 8 of 9 the heading "Do not recycle:" was still written, followed by unrelated context text (a disposal sentence or a composition line), which can mislead ("Do not recycle: Plastic bags can be returned to supermarket collection points"). Only Miscellaneous Trash left the empty section out. Word-overlap checks cannot catch this because every sentence is in the context.

## Step 13: Does Retrieval Supply the Chunks the Model Needs?

Retrieve inside the requested category (plus the general chunk) with the hybrid score, for k = 3 to 6, and count how many of the needed chunks are found.

In [ ]:
def get_chunks(question, category, k=6, alpha=0.25):
    scores = hybrid_scores(question, alpha)
    allowed = np.isin(chunk_category, [category, "ALL"])
    scores = np.where(allowed, scores, -np.inf)
    best = np.argsort(-scores)[:k]
    return [rag_corpus.iloc[i] for i in best]

rows = []
for category in CATEGORIES:
    needed_ids = {row["chunk_id"] for row in needed_chunks(category)}
    for k in (3, 4, 5, 6):
        question = f"How to recycle {category}"
        found = get_chunks(question, category, k)
        found_ids = {row["chunk_id"] for row in found}
        n_tokens = len(tok(make_input(question, found)).input_ids)
        rows.append({"category": category, "k": k, "needed found": len(needed_ids & found_ids),
                     "needed": len(needed_ids), "tokens": n_tokens})
table = pd.DataFrame(rows)
table["complete"] = table["needed found"] == table["needed"]
print(table.pivot(index="category", columns="k", values="needed found"))
print(table.groupby("k").agg(share_complete=("complete", "mean"), max_tokens=("tokens", "max")))

**Result:** All needed chunks are found for 0 of 9 categories at k = 3 and k = 4, 4 of 9 at k = 5, and 9 of 9 at k = 6 (longest input 359 tokens, inside the 512 limit). A category has only 6 chunks, so k = 6 is effectively the whole category: with the category given, retrieval is a filter, and ranking matters more when the category is unknown. The model now also sees the category's Acceptable Items and Benefits chunks, which were never in its training context.

## Step 14: The Whole Pipeline on All Nine Categories

In [ ]:
def token_f1(a, b):
    count_a, count_b = collections.Counter(a.lower().split()), collections.Counter(b.lower().split())
    common = sum((count_a & count_b).values())
    if common == 0:
        return 0.0
    precision, recall = common / sum(count_a.values()), common / sum(count_b.values())
    return 2 * precision * recall / (precision + recall)

outputs, rows = {}, []
for category in CATEGORIES:
    question = f"How to recycle {category}"
    answer = generate_answer(final_model, question, get_chunks(question, category, k=6))
    outputs[category] = answer
    reference = references[category]
    extra = sorted(set(answer.lower().split()) - set(reference.lower().split()))[:8]
    missing = sorted(set(reference.lower().split()) - set(answer.lower().split()))[:8]
    rows.append({"category": category, "exact": answer.strip() == reference.strip(),
                 "token F1": round(token_f1(answer, reference), 3), "extra words": extra, "missing words": missing})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** All 9 answers match the reference exactly (token F1 1.0, no extra or missing words), even with the extra Acceptable Items and Benefits chunks in the context. Together with the edit test (9 of 9) and the removal test (0 invented items) this shows the model uses its context. These are categories seen in training, so the unseen-category test is still needed.

## Step 15: Sampling Methods and Parameters

Greedy, beam search and four kinds of random sampling. Trigram support is the share of three-word sequences in the answer that appear in the retrieved text (the references themselves score below 1.0 because of template words like "Before recycling:", so that is the ceiling).

In [ ]:
def trigram_support(answer, context):
    words = answer.lower().split()
    grams = [tuple(words[i:i + 3]) for i in range(len(words) - 2)]
    if not grams:
        return 1.0
    context_words = context.lower().split()
    context_grams = {tuple(context_words[i:i + 3]) for i in range(len(context_words) - 2)}
    return sum(g in context_grams for g in grams) / len(grams)

settings = {
    "greedy": dict(do_sample=False, num_beams=1),
    "beam search (4)": dict(do_sample=False, num_beams=4, early_stopping=True),
    "sampling T=0.7": dict(do_sample=True, temperature=0.7, top_k=0),
    "sampling T=1.0": dict(do_sample=True, temperature=1.0, top_k=0),
    "top-p 0.9, T=1.0": dict(do_sample=True, temperature=1.0, top_p=0.9, top_k=0),
    "top-k 20, T=1.0": dict(do_sample=True, temperature=1.0, top_k=20),
}
seeds = [1, 2]
records = []
for name, options in settings.items():
    for category in CATEGORIES:
        question = f"How to recycle {category}"
        chunk_rows = get_chunks(question, category, k=6)
        context = " ".join(" ".join(row["text"].split()) for row in chunk_rows)
        for seed in (seeds if options["do_sample"] else [0]):
            torch.manual_seed(seed)
            answer = generate_answer(final_model, question, chunk_rows, **options)
            records.append({"setting": name, "exact": answer.strip() == references[category].strip(),
                            "token F1": token_f1(answer, references[category]),
                            "trigram support": trigram_support(answer, context), "words": len(answer.split())})
    print("done:", name, flush=True)

summary = pd.DataFrame(records).groupby("setting", sort=False).mean().round(3)
print(summary)

**Result:** Greedy, beam search, T = 0.7 and top-p 0.9 reproduce the references exactly (trigram support 0.634, the ceiling). Pure sampling at T = 1.0 and top-k 20 each gave one non-identical answer out of 18 (exact 0.944, trigram support 0.629). The model puts almost all its probability on one answer, so settings barely matter. Choice: greedy decoding, which is deterministic, as accurate as the rest, and about 4 times cheaper than beam search. Only 9 categories and 2 seeds.

## Step 16: The Answer That Deviated

In [ ]:
for name in ["sampling T=1.0", "top-k 20, T=1.0"]:
    for category in CATEGORIES:
        question = f"How to recycle {category}"
        chunk_rows = get_chunks(question, category, k=6)
        for seed in seeds:
            torch.manual_seed(seed)
            answer = generate_answer(final_model, question, chunk_rows, **settings[name])
            if answer.strip() != references[category].strip():
                extra = [w for w in answer.split() if w not in set(references[category].split())]
                print(name, "|", category, "| seed", seed)
                print("  extra words:", extra)
                print("  ANSWER:", answer, "\n")

**Result:** In Paper (T = 1.0 and top-k 20, seed 2) the model wrote a second "Do not recycle:" heading and listed under it "Office paper and mail; Newspapers and magazines; Books and catalogs; Paper bags; Wrapping paper; Shredded paper", which are the Paper policy's Acceptable Items. The answer tells a resident not to recycle what the policy accepts. Every word is in the context, so nothing was invented, and word overlap barely moved (trigram support 0.634 to 0.629). A faithfulness check has to know which list belongs under which heading.

## Step 17: The Section-Aware Checker and the Final Function

Three rules: no heading twice; every bullet of the retrieved chunks appears only under the heading of its own source (Acceptable Items bullets must not appear at all); and a heading may not appear if its source chunk was not retrieved. An answer that fails is rebuilt directly from the policy text.

In [ ]:
HEADINGS = ["Before recycling:", "Do not recycle:", "Cannot be recycled:", "Special handling:", "Note:"]
SOURCE_HEADING = {"Preparation Instructions": "Before recycling:", "Non-Acceptable Items": "Do not recycle:",
                  "Items That Cannot Be Recycled": "Cannot be recycled:", "Special Handling Items": "Special handling:"}

def check_answer(answer, chunk_rows):
    problems = []
    for heading in HEADINGS:                                       # rule 1
        if answer.count(heading) > 1:
            problems.append(f"heading repeated: {heading}")

    positions = sorted((m.start(), h) for h in HEADINGS for m in re.finditer(re.escape(h), answer))
    def heading_before(index):
        earlier = [h for p, h in positions if p < index]
        return earlier[-1] if earlier else None

    for row in chunk_rows:                                         # rule 2
        expected = SOURCE_HEADING.get(row["header"])
        for bullet in bullets_of(row["text"].split("\n", 1)[1]):
            for match in re.finditer(re.escape(bullet), answer):
                found_under = heading_before(match.start())
                if expected is None:
                    problems.append(f"'{bullet}' (from {row['header']}) should not appear")
                elif found_under != expected:
                    problems.append(f"'{bullet}' is under '{found_under}', expected '{expected}'")

    for heading in HEADINGS[:4]:                                   # rule 3
        has_source = any(SOURCE_HEADING.get(row["header"]) == heading for row in chunk_rows)
        if heading in answer and not has_source:
            problems.append(f"{heading} written but its source chunk was not retrieved")
    return problems

In [ ]:
def generate_instructions(category, question=None, k=6):
    matches = [c for c in CATEGORIES if c.lower() == str(category).strip().lower()]
    if not matches:
        return {"category": category, "answer": None, "status": "unknown category", "problems": [], "sources": []}
    category = matches[0]
    question = question or f"How to recycle {category}"
    chunk_rows = get_chunks(question, category, k)
    raw = generate_answer(final_model, question, chunk_rows)             # greedy decoding
    problems = check_answer(raw, chunk_rows)
    if problems:
        answer, status = reference_answer(category), "fallback (failed the check; rebuilt from the sources)"
    else:
        answer, status = raw, "ok"
    return {"category": category, "answer": answer, "status": status, "problems": problems,
            "sources": [row["chunk_id"] for row in chunk_rows], "raw_model_output": raw}

for category in CATEGORIES:
    print(category, "->", generate_instructions(category)["status"])
print(generate_instructions("glass")["status"])
print(generate_instructions("Furniture")["status"])

In [ ]:
# the inverted Paper answer must be caught and replaced
paper_bad = ("To recycle Paper: Place in paper recycling bins or mixed recycling where accepted. Before recycling: Keep paper dry and clean; "
             "Remove plastic wrappings, windows, or bindings; Shred confidential documents. Do not recycle: Food-soiled paper; Waxed paper; "
             "Paper towels and napkins; Thermal receipt paper; Paper with plastic coatings. Do not recycle: Office paper and mail; "
             "Newspapers and magazines; Books and catalogs; Paper bags; Wrapping paper (non-metallic); Shredded paper (in paper bags). "
             "Ensure it is clean and dry before recycling.")
paper_chunks = get_chunks("How to recycle Paper", "Paper", 6)
problems = check_answer(paper_bad, paper_chunks)
print(len(problems), "problems, for example:", problems[:2])

**Result:** All 9 categories give status "ok", lowercase "glass" is accepted, and "Furniture" returns an unknown-category message. The inverted Paper answer is caught (repeated heading, Acceptable Items under "Do not recycle:") and replaced by the rebuilt answer. The checker does not look at the collection sentence, the card sentences or the final note, and it cannot detect a missing section. Honest note: the references are templated, so the rebuild alone gives the right answers for this data; the model's value is that it follows edited facts and does not invent, with its occasional failures caught.

## Step 18: Unseen Category: Paper Held Out

Train a new model from scratch on the eight other categories, then ask it for Paper with Paper's real retrieved chunks.

In [ ]:
train_heldout = train_small[train_small["category"] != "Paper"]
val_heldout = val_examples[val_examples["category"] != "Paper"]
print(len(train_heldout), len(val_heldout), "| Paper in training:", (train_heldout["category"] == "Paper").any())

start = time.time()
heldout_model, heldout_history = finetune(train_heldout, val_heldout, epochs=3)
print(heldout_history.to_string(index=False))
print("minutes:", round((time.time() - start) / 60, 1))

In [ ]:
question = "How to recycle Paper"
paper_chunks = get_chunks(question, "Paper", k=6)
answer = generate_answer(heldout_model, question, paper_chunks)
print("exact match:", answer.strip() == references["Paper"].strip(), "| token F1:", round(token_f1(answer, references["Paper"]), 3))
print("checker problems:", check_answer(answer, paper_chunks))
print("\nANSWER:", answer)

# how much Paper text did training contain?
has_paper_chunk = (train_heldout["input"].str.contains("Paper Recycling Guidelines") |
                   train_heldout["input"].str.contains("Paper | Disposal card", regex=False))
print("\ntraining inputs containing a Paper chunk:", int(has_paper_chunk.sum()), "of", len(train_heldout))
print("training targets that are a Paper answer:", int(train_heldout["target"].str.contains("To recycle Paper:", regex=False).sum()))

**Result:** 17.4 minutes; validation loss 0.0285, 0.0021, 0.0029. The new model wrote the complete correct Paper answer (exact match, token F1 1.0, no checker problems) without ever being trained to produce one. 17 of the 192 training inputs showed a Paper chunk as a distractor, but 0 targets were Paper answers, and the loss is computed only on the answer, so it never learned to write Paper text. Caveats: one category, one run; Paper shares its five-section layout with seven others. A category with a different layout (Miscellaneous Trash) would be a harder test and was not run.

## Step 19: Readability

In [ ]:
# pip install textstat    (once, if missing)
import textstat

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def near_duplicate_pairs(text, threshold=0.5):
    word_sets = [set(re.findall(r"[a-z]+", s.lower())) for s in split_sentences(text)]
    pairs = 0
    for i in range(len(word_sets)):
        for j in range(i + 1, len(word_sets)):
            union = word_sets[i] | word_sets[j]
            if union and len(word_sets[i] & word_sets[j]) / len(union) >= threshold:
                pairs += 1
    return pairs

rows = []
for category in CATEGORIES:
    answer = outputs[category]
    sentences = split_sentences(answer)
    rows.append({"category": category, "words": len(answer.split()), "sentences": len(sentences),
                 "words per sentence": round(len(answer.split()) / len(sentences), 1),
                 "Flesch ease": round(textstat.flesch_reading_ease(answer), 1),
                 "grade level": round(textstat.flesch_kincaid_grade(answer), 1),
                 "near-duplicate pairs": near_duplicate_pairs(answer)})
readability = pd.DataFrame(rows)
print(readability.to_string(index=False))
print(readability[["words per sentence", "Flesch ease", "grade level", "near-duplicate pairs"]].mean().round(2).to_dict())

**Result:** Average Flesch reading ease 39.9 (difficult; plain English is 60 or more), grade level 10.3, 10.5 words per sentence. Sentences are short, so the low score comes from long words (recycling, containers, hazardous). Miscellaneous Trash is hardest (11.1, grade 14.4); Food Organics easiest (54.2). Only one near-duplicate sentence pair. Flesch is built for prose, not lists of nouns, so this is a rough guide, and the model cannot be more readable than the policy text it copies.

## Summary

- **System:** category in, hybrid retrieval (TF-IDF weight 0.25 plus MiniLM) inside the category's 6 chunks, fine-tuned `flan-t5-small` with greedy decoding, section-aware checker, rebuild from the sources if it fails. The function is `generate_instructions(category)`.
- **Evidence it uses the documents:** all 9 references reproduced; the edited fact was used in 9 of 9; 0 invented items when a section was removed; a model trained without Paper still wrote the exact Paper answer.
- **What went wrong:** with a section missing the model fills the slot with misplaced text (8 of 9), and under sampling it once listed Paper's accepted items under "Do not recycle:". Word overlap could not see it; the checker does.
- **Limitations:** the references are templated, so the model's value is robustness, not capability; one held-out category sharing a layout with seven others; a small hand-written retrieval set; the checker misses the collection sentence, card sentences and the note; readability is inherited from the policy text; single runs on a CPU.

# Part 5: Integrated Waste Management Assistant

One assistant that takes a photo, a description, or both, and returns the waste category, recycling instructions and the policy text they came from. It joins the three models:

- the CNN from Part 2 (`waste_cnn.keras`),
- the text classifier from Part 3 (rebuilt here in seconds),
- the instruction generator from Part 4 (the folder `waste_rag_model`).

Needs next to this notebook: `waste_cnn.keras`, the folder `waste_rag_model`, the folder `RealWaste`, `waste_descriptions.csv` and `waste_policy_documents.json`. TensorFlow and PyTorch are both used. Close other notebooks first, because both models sit in memory. Numbers in the Results come from my runs.

## Step 1: Loading Libraries

In [ ]:
import os
os.environ["USE_TF"] = "0"          # keep transformers from loading TensorFlow as well
import re
import json
import time
import uuid
import datetime
import tempfile
import numpy as np
import pandas as pd
import torch                         # imported before TensorFlow (the order that works on this machine)
import tensorflow as tf
from tensorflow import keras
from PIL import Image

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score

np.random.seed(42)

## Step 2: Check the Files and Load the Saved Models

In [ ]:
for name in ["waste_cnn.keras", "waste_rag_model", "RealWaste", "waste_descriptions.csv", "waste_policy_documents.json"]:
    print(name, "->", "found" if os.path.exists(name) else "MISSING")

cnn = keras.models.load_model("waste_cnn.keras", compile=False)
print("CNN output shape:", cnn.output_shape)

tok = AutoTokenizer.from_pretrained("waste_rag_model")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("waste_rag_model").float().eval()
print("generator parameters:", sum(p.numel() for p in gen_model.parameters()))

**Result:** All five files are found. The CNN outputs 9 probabilities and the generator has 76,961,152 parameters. PyTorch and TensorFlow load together in one process. transformers warns that two weight tables in the saved config differ and will not be tied; the Glass check in Step 8 shows the reload kept the fine-tuned behaviour.

## Step 3: The Held-Out Images

The 950 images the CNN never trained on, rebuilt with the same split settings as Part 1. The file path holds the class folder, so the true label can be read from it. We read images from these paths, not from the dataset's iteration order (that order caused the overlap bug in Part 1).

In [ ]:
holdout = keras.utils.image_dataset_from_directory(
    "RealWaste", validation_split=0.2, subset="validation", seed=42,
    image_size=(224, 224), batch_size=32, label_mode="categorical", shuffle=True)
paths = holdout.file_paths
print("hold-out images:", len(paths))
print(paths[:3])

**Result:** 950 images, matching Part 1. About half of them were the validation set that chose the tuned CNN, so scores on them are slightly optimistic. The exact 475 test images of Part 2 cannot be recovered, because their file names were not kept.

## Step 4: Policy Chunks

Same corpus as Part 4: category documents, one copy of the general rules, and one disposal card per category.

In [ ]:
desc = pd.read_csv("waste_descriptions.csv")
policies = json.load(open("waste_policy_documents.json"))
header_pattern = re.compile(r"^([A-Za-z][A-Za-z \-/&]+):\s*$", re.M)

def split_sections(text):
    parts = header_pattern.split(text)
    return list(zip(parts[1::2], [b.strip() for b in parts[2::2]]))

chunks = []
for policy in policies:
    for header, body in split_sections(policy["document_text"]):
        categories = policy["categories_covered"]
        match = re.match(r"([A-Z ]+) GUIDELINES$", header)
        if len(categories) > 1:
            if header == "GENERAL REQUIREMENTS":
                categories = ["ALL"]
            elif match:
                own = [c for c in categories if c.upper() == match.group(1).strip()]
                categories = own or categories
        chunks.append({"chunk_id": f"policy{policy['policy_id']}::{header}", "policy_id": policy["policy_id"],
                       "header": header, "category": categories[0],
                       "text": f"{policy['policy_type']} | {header}\n{body}"})
chunk_table = pd.DataFrame(chunks)

In [ ]:
own_docs = chunk_table[chunk_table["policy_id"] <= 9]
general_chunk = chunk_table[chunk_table["header"] == "GENERAL REQUIREMENTS"].head(1)

cards = []
for category, group in desc.groupby("category"):
    instructions = sorted(group["disposal_instruction"].dropna().unique())
    note = group["common_confusion"].dropna().unique()
    text = (f"{category} | Disposal card\nComposition: {group['material_composition'].iloc[0]}\n"
            f"Disposal: " + " ".join(instructions))
    if len(note) > 0:
        text += "\nCommon confusion: " + note[0]
    cards.append({"chunk_id": f"card::{category}", "policy_id": None, "header": "Disposal card",
                  "category": category, "text": text})

rag_corpus = pd.concat([own_docs, general_chunk, pd.DataFrame(cards)], ignore_index=True)
CATEGORIES = sorted(desc["category"].unique())
general_row = rag_corpus[rag_corpus["category"] == "ALL"].iloc[0]
print("chunks:", len(rag_corpus))

## Step 5: Retrieval

Hybrid search (TF-IDF weight 0.25 plus MiniLM embeddings), restricted to the requested category and the general chunk.

In [ ]:
chunk_category = rag_corpus["category"].values
tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
tfidf_matrix = tfidf.fit_transform(rag_corpus["text"].tolist())
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
dense_matrix = encoder.encode(rag_corpus["text"].tolist(), normalize_embeddings=True, show_progress_bar=False)

def hybrid_scores(question, alpha=0.25):
    a = cosine_similarity(tfidf.transform([question]), tfidf_matrix)[0]
    b = dense_matrix @ encoder.encode([question], normalize_embeddings=True)[0]
    a = (a - a.mean()) / (a.std() + 1e-9)
    b = (b - b.mean()) / (b.std() + 1e-9)
    return alpha * a + (1 - alpha) * b

def get_chunks(question, category, k=6):
    scores = hybrid_scores(question)
    scores = np.where(np.isin(chunk_category, [category, "ALL"]), scores, -np.inf)
    best = np.argsort(-scores)[:k]
    return [rag_corpus.iloc[i] for i in best]

## Step 6: The Instruction Generator and Its Checker

The same functions as Part 4: build the answer for a category, check it section by section, rebuild it from the sources if the check fails.

In [ ]:
def chunk_text(category, header):
    row = rag_corpus[(rag_corpus["category"] == category) & (rag_corpus["header"] == header)]
    if row.empty:
        return ""
    return row["text"].iloc[0].split("\n", 1)[1].strip()

def bullets_of(body):
    return [line[2:].strip() for line in body.splitlines() if line.startswith("- ")]

def prose_of(body):
    lines = [line.strip() for line in body.splitlines() if line.strip() and not line.startswith("- ")]
    return " ".join(lines)

def reference_answer(category):
    card = chunk_text(category, "Disposal card")
    disposal = card.split("Disposal: ")[1].split("\nCommon confusion:")[0].strip()
    note = ""
    if "Common confusion: " in card:
        note = card.split("Common confusion: ")[1].strip()
    if category == "Miscellaneous Trash":
        cannot = bullets_of(chunk_text(category, "Items That Cannot Be Recycled"))
        special = bullets_of(chunk_text(category, "Special Handling Items"))
        text = (f"{category} cannot be recycled as ordinary material. Cannot be recycled: {'; '.join(cannot)}. "
                f"Special handling: {'; '.join(special)}. {disposal}")
    else:
        collect = prose_of(chunk_text(category, "Collection Method"))
        prepare = bullets_of(chunk_text(category, "Preparation Instructions"))
        avoid = bullets_of(chunk_text(category, "Non-Acceptable Items"))
        text = (f"To recycle {category}: {collect} Before recycling: {'; '.join(prepare)}. "
                f"Do not recycle: {'; '.join(avoid)}. {disposal}")
    if note:
        text += " Note: " + note
    return text.strip()

references = {c: reference_answer(c) for c in CATEGORIES}

In [ ]:
def make_input(question, chunk_rows):
    context = " ".join(" ".join(row["text"].split()) for row in chunk_rows)
    return f"Answer the question using only the context. Question: {question} Context: {context}"

def generate_answer(question, chunk_rows):
    inputs = tok(make_input(question, chunk_rows), return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output = gen_model.generate(**inputs, max_new_tokens=200, num_beams=1)
    return tok.decode(output[0], skip_special_tokens=True)

In [ ]:
HEADINGS = ["Before recycling:", "Do not recycle:", "Cannot be recycled:", "Special handling:", "Note:"]
SOURCE_HEADING = {"Preparation Instructions": "Before recycling:", "Non-Acceptable Items": "Do not recycle:",
                  "Items That Cannot Be Recycled": "Cannot be recycled:", "Special Handling Items": "Special handling:"}

def check_answer(answer, chunk_rows):
    problems = []
    for heading in HEADINGS:                                       # rule 1: no heading twice
        if answer.count(heading) > 1:
            problems.append(f"heading repeated: {heading}")

    positions = sorted((m.start(), h) for h in HEADINGS for m in re.finditer(re.escape(h), answer))
    def heading_before(index):
        earlier = [h for p, h in positions if p < index]
        return earlier[-1] if earlier else None

    for row in chunk_rows:                                         # rule 2: bullets under their own heading
        expected = SOURCE_HEADING.get(row["header"])
        for bullet in bullets_of(row["text"].split("\n", 1)[1]):
            for match in re.finditer(re.escape(bullet), answer):
                found_under = heading_before(match.start())
                if expected is None:
                    problems.append(f"'{bullet}' (from {row['header']}) should not appear")
                elif found_under != expected:
                    problems.append(f"'{bullet}' is under '{found_under}', expected '{expected}'")

    for heading in HEADINGS[:4]:                                   # rule 3: heading needs its source chunk
        has_source = any(SOURCE_HEADING.get(row["header"]) == heading for row in chunk_rows)
        if heading in answer and not has_source:
            problems.append(f"{heading} written but its source chunk was not retrieved")
    return problems

In [ ]:
answer_cache = {}          # the answer depends only on the category, so we keep it

def generate_instructions(category, question=None, k=6):
    matches = [c for c in CATEGORIES if c.lower() == str(category).strip().lower()]
    if not matches:
        return {"category": category, "answer": None, "status": "unknown category", "problems": [], "sources": []}
    category = matches[0]
    question = question or f"How to recycle {category}"
    key = (category, question, k)
    if key in answer_cache:
        return answer_cache[key]

    chunk_rows = get_chunks(question, category, k)
    raw = generate_answer(question, chunk_rows)
    problems = check_answer(raw, chunk_rows)
    if problems:
        answer, status = reference_answer(category), "fallback (failed the check; rebuilt from the sources)"
    else:
        answer, status = raw, "ok"
    result = {"category": category, "answer": answer, "status": status, "problems": problems,
              "sources": [{"chunk_id": row["chunk_id"], "text": row["text"]} for row in chunk_rows]}
    answer_cache[key] = result
    return result

## Step 7: The Text Classifier

Rebuilt from the CSV with the Part 3 settings: same cleaning, same split and seed, TF-IDF with word pairs and a calibrated LinearSVC, fitted on training and validation descriptions only.

In [ ]:
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_data = desc[["description", "category"]].copy()
text_data["clean"] = text_data["description"].apply(clean_text)
text_data = text_data.drop_duplicates(subset="clean").reset_index(drop=True)
LABELS = sorted(text_data["category"].unique())
label_to_id = {label: i for i, label in enumerate(LABELS)}
text_data["y"] = text_data["category"].map(label_to_id)

train_data, hold_data = train_test_split(text_data, test_size=0.20, stratify=text_data["y"], random_state=42)
val_data, test_data = train_test_split(hold_data, test_size=0.50, stratify=hold_data["y"], random_state=42)

dev_text = pd.concat([train_data["clean"], val_data["clean"]])
dev_y = pd.concat([train_data["y"], val_data["y"]])

def new_text_model():
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
                     ("clf", CalibratedClassifierCV(LinearSVC(), cv=5))])

text_model = new_text_model()
text_model.fit(dev_text, dev_y)
print("text model test accuracy:", round(accuracy_score(test_data["y"], text_model.predict(test_data["clean"])), 4))

In [ ]:
CONFIDENCE_THRESHOLD = 0.40

def predict_waste_category(text, threshold=CONFIDENCE_THRESHOLD):
    if not isinstance(text, str) or clean_text(text) == "":
        return {"category": None, "confidence": 0.0, "status": "empty input", "top3": []}
    cleaned = clean_text(text)
    word_numbers = text_model.named_steps["tfidf"].transform([cleaned])
    if word_numbers.nnz == 0:
        return {"category": None, "confidence": 0.0, "status": "no known words", "top3": []}
    probs = text_model.predict_proba([cleaned])[0]
    order = np.argsort(-probs)[:3]
    top3 = [(LABELS[i], round(float(probs[i]), 2)) for i in order]
    status = "ok" if probs[order[0]] >= threshold else "low confidence"
    return {"category": LABELS[order[0]], "confidence": round(float(probs[order[0]]), 2),
            "status": status, "top3": top3}

## Step 8: Smoke Test of the Three Models

In [ ]:
def load_for_cnn(path):
    image = keras.utils.load_img(path, target_size=(224, 224))
    return keras.utils.img_to_array(image)[None, ...]        # shape (1, 224, 224, 3), values 0-255

true_label = os.path.basename(os.path.dirname(paths[0]))
probs = cnn.predict(load_for_cnn(paths[0]), verbose=0)[0]
print("image:", paths[0], "| true:", true_label, "| CNN says:", CATEGORIES[int(probs.argmax())], round(float(probs.max()), 2))

print("text:", predict_waste_category("red glass pen"))

glass = generate_instructions("Glass")
print("generator status:", glass["status"], "| matches the Part 4 reference:", glass["answer"] == references["Glass"])
print("sources:", [s["chunk_id"] for s in glass["sources"]])

**Result:** Text model test accuracy 1.0, as in Part 3; "red glass pen" (wrong in two earlier models) is Miscellaneous Trash at 0.91. The first hold-out image (truly Plastic) is classified Plastic at confidence 0.65. The reloaded generator writes the Glass answer identical to the Part 4 reference, using the six Glass chunks as sources (the general chunk is added by the assistant in Step 11).

## Step 9: How Reliable Is the CNN's Confidence?

We run the CNN over all 950 hold-out images (reading images from their paths), and compare its confidence with whether it was right.

In [ ]:
def cnn_probabilities(path_list, batch_size=32):
    batches = []
    for i in range(0, len(path_list), batch_size):
        images = np.concatenate([load_for_cnn(p) for p in path_list[i:i + batch_size]])
        batches.append(cnn.predict(images, verbose=0))
    return np.concatenate(batches)

true_numbers = np.array([CATEGORIES.index(os.path.basename(os.path.dirname(p))) for p in paths])
start = time.time()
cnn_probs = cnn_probabilities(paths)
seconds = time.time() - start
predicted = cnn_probs.argmax(1)
confidence = cnn_probs.max(1)
print("seconds:", round(seconds), "| per image:", round(seconds / len(paths), 3))
print("hold-out accuracy:", round(float((predicted == true_numbers).mean()), 4))

rows = []
for low, high in [(0.0, 0.5), (0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)]:
    in_band = (confidence >= low) & (confidence < high)
    rows.append({"confidence band": f"{low:.1f}-{min(high, 1):.1f}", "images": int(in_band.sum()),
                 "accuracy": round(float((predicted[in_band] == true_numbers[in_band]).mean()), 3)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# pictures that are not waste
random_state = np.random.RandomState(0)
synthetic = {
    "black": np.zeros((1, 224, 224, 3), "float32"),
    "white": np.full((1, 224, 224, 3), 255, "float32"),
    "random noise": random_state.randint(0, 256, (1, 224, 224, 3)).astype("float32"),
    "gradient": np.tile(np.linspace(0, 255, 224, dtype="float32")[None, None, :, None], (1, 224, 1, 3)),
}
for name, image in synthetic.items():
    probs = cnn.predict(image, verbose=0)[0]
    print(f"{name:14} -> {CATEGORIES[int(probs.argmax())]:20} confidence {probs.max():.2f}")

**Result:** 30 seconds for 950 images (0.032 s each). Hold-out accuracy 0.800. By confidence band: 0.0-0.5: 0.514 (70 images); 0.5-0.6: 0.543 (105); 0.6-0.7: 0.554 (83); 0.7-0.8: 0.690 (100); 0.8-1.0: 0.932 (592). Below 0.7 the CNN is close to a coin flip, with steps up at 0.7 and 0.8. Non-waste pictures: black gets Metal at 0.45, white Cardboard at 0.28, noise Plastic at 0.57 (0.54 after resizing in the assistant), gradient Cardboard at 0.51, so the CNN cannot reject non-waste images itself. Decision, cut-offs 0.80 and 0.70: at least 0.80 answer directly (62% of images, 93% accurate); 0.70 to 0.80 answer and ask to confirm (10.5%, 69%); below 0.70 (27%) ask for a description or a clearer photo. A first guess of 0.50 and 0.70 was dropped because noise at 0.54 was still answered and 0.5 to 0.7 is no more reliable than below. The cut-offs were set on the same 950 images and each low band has only 70 to 105 images, so they are rough.

## Step 10: The Assistant

`assist(image_path, text)` checks each input, classifies the photo and/or the description, and decides: one usable input gives an answer; agreeing inputs give an answer; disagreeing inputs give a conflict (the user confirms, instructions for both are shown); nothing usable gives "need more information".

In [ ]:
IMG_CONFIDENT, IMG_UNCERTAIN = 0.80, 0.70
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
MAX_TEXT_CHARS = 1000

def classify_image(path):
    start = time.time()
    empty = {"category": None, "confidence": 0.0, "top3": [], "seconds": 0.0}
    try:
        path = str(path)
        if not os.path.exists(path):
            return {**empty, "status": "error: file not found"}
        if os.path.splitext(path)[1].lower() not in IMAGE_EXTENSIONS:
            return {**empty, "status": "error: unsupported file type (use .jpg or .png)"}
        probs = cnn.predict(load_for_cnn(path), verbose=0)[0]
    except Exception as error:                              # unreadable or corrupt file
        return {**empty, "status": f"error: cannot read the image ({type(error).__name__})"}
    order = np.argsort(-probs)[:3]
    top = float(probs[order[0]])
    level = "confident" if top >= IMG_CONFIDENT else "uncertain" if top >= IMG_UNCERTAIN else "unsure"
    return {"status": level, "category": CATEGORIES[order[0]], "confidence": round(top, 2),
            "top3": [(CATEGORIES[i], round(float(probs[i]), 2)) for i in order], "seconds": round(time.time() - start, 3)}

In [ ]:
def classify_text(text):
    start = time.time()
    if not isinstance(text, str):
        return {"category": None, "confidence": 0.0, "top3": [], "status": "error: the description must be text", "seconds": 0.0}
    result = predict_waste_category(text[:MAX_TEXT_CHARS])
    result["seconds"] = round(time.time() - start, 3)
    return result

def usable_guesses(image_result, text_result):
    guesses = []
    if image_result and image_result["status"] in ("confident", "uncertain"):
        guesses.append({"source": "image", "category": image_result["category"], "level": image_result["status"]})
    if text_result and text_result["status"] == "ok":
        guesses.append({"source": "text", "category": text_result["category"], "level": "confident"})
    return guesses

In [ ]:
def need_more_result(image_result, text_result):
    reasons, options = [], []
    if image_result:
        reasons.append("photo: " + image_result["status"])
        options = image_result["top3"]
    if text_result:
        reasons.append("description: " + text_result["status"])
        options = options or text_result["top3"]
    message = "I could not tell what this is (" + "; ".join(reasons) + "). Please add a short description or a clearer photo."
    return {"status": "need more information", "needs_confirmation": True, "options": options, "message": message}

def conflict_result(guesses):
    message = (f"The photo suggests {guesses[0]['category']} but the description suggests {guesses[1]['category']}. "
               "Please confirm which one it is. Instructions for both are shown.")
    return {"status": "conflict", "needs_confirmation": True, "message": message}

def answered_result(category, guesses, image_result, text_result):
    uncertain = len(guesses) == 1 and guesses[0]["level"] == "uncertain"
    used = " and ".join(g["source"] for g in guesses)
    message = f"Based on the {used}: {category}."
    if uncertain:
        message += " I am not fully sure, please check."
    if image_result and image_result["status"] not in ("confident", "uncertain"):
        message += " (the photo was too unclear to use)"
    if text_result and text_result["status"] != "ok":
        message += " (the description was too unclear to use)"
    status = "answered, please confirm" if uncertain else "answered"
    return {"status": status, "needs_confirmation": uncertain, "category": category, "message": message}

def add_instructions(result, categories):
    for category in categories:
        answer = generate_instructions(category)
        result["instructions"].append({"category": category, "answer": answer["answer"],
                                       "status": answer["status"], "problems": answer["problems"]})
        known = [s["chunk_id"] for s in result["sources"]]
        result["sources"] += [s for s in answer["sources"] if s["chunk_id"] not in known]
    result["sources"].append({"chunk_id": general_row["chunk_id"], "text": general_row["text"]})

In [ ]:
request_log = {}

def assist(image_path=None, text=None):
    start = time.time()
    request_id = uuid.uuid4().hex[:8]
    has_image = image_path not in (None, "")
    has_text = text is not None and not (isinstance(text, str) and text.strip() == "")
    image_result = classify_image(image_path) if has_image else None
    text_result = classify_text(text) if has_text else None
    guesses = usable_guesses(image_result, text_result)

    result = {"request_id": request_id, "image": image_result, "text": text_result, "instructions": [],
              "sources": [], "needs_confirmation": False, "input_text": text if isinstance(text, str) else None}

    if image_result is None and text_result is None:
        result.update(status="no input", message="Please give me a photo or a short description of the item.")
    elif len(guesses) == 0:
        result.update(need_more_result(image_result, text_result))
    else:
        categories = sorted({g["category"] for g in guesses})
        if len(categories) > 1:
            result.update(conflict_result(guesses))
        else:
            result.update(answered_result(categories[0], guesses, image_result, text_result))
        add_instructions(result, categories)

    result["seconds"] = round(time.time() - start, 3)
    request_log[request_id] = result
    return result

def show(result):
    print(f"[{result['request_id']}] {result['status']} ({result['seconds']} s) - {result['message']}")
    for item in result["instructions"]:
        print(f"  {item['category']} [{item['status']}]: {item['answer'][:110]}...")
    if result["sources"]:
        print("  policy sources:", [s["chunk_id"] for s in result["sources"]])

## Step 11: Edge Cases, Speed and a Real Test Image

The answer for each category is generated once and cached, so we warm the cache first and time it. Then cases that should not crash it.

In [ ]:
def first_path(category):
    for p in paths:
        if os.path.basename(os.path.dirname(p)) == category:
            return p

start = time.time()
for category in CATEGORIES:
    generate_instructions(category)
print("warm-up seconds (9 categories):", round(time.time() - start))

cases = [
    ("description only", dict(text="empty green glass bottle with contents")),
    ("no input", dict()),
    ("whitespace text", dict(text="   ")),
    ("missing file", dict(image_path="does_not_exist.jpg")),
    ("wrong file type", dict(image_path="waste_descriptions.csv")),
    ("gibberish", dict(text="asdf qwerty")),
    ("other language", dict(text="chupa ya plastiki")),
    ("text that is not a string", dict(text=123)),
]
rows = []
for name, arguments in cases:
    answer = assist(**arguments)
    rows.append({"case": name, "status": answer["status"], "category": answer.get("category"), "seconds": answer["seconds"]})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
with tempfile.TemporaryDirectory() as folder:                  # temporary files, deleted automatically
    black_path = os.path.join(folder, "black.png")
    Image.new("RGB", (300, 300), (0, 0, 0)).save(black_path)
    noise_path = os.path.join(folder, "noise.png")
    noise = np.random.RandomState(0).randint(0, 256, (300, 300, 3), dtype="uint8")
    Image.fromarray(noise).save(noise_path)
    for name, p in [("black image", black_path), ("noise image", noise_path)]:
        answer = assist(image_path=p)
        print(name, "->", answer["status"], "| CNN confidence", answer["image"]["confidence"])

for name, p in [("Plastic hold-out", first_path("Plastic")), ("Glass hold-out", first_path("Glass"))]:
    answer = assist(image_path=p)
    print(name, "->", answer["status"], "| CNN confidence", answer["image"]["confidence"])

# a description rescues an unclear photo
answer = assist(image_path=first_path("Plastic"), text="plastic bottle")
print("\nPlastic photo + 'plastic bottle':", answer["status"], "|", answer["message"])
answer = assist(image_path=first_path("Plastic"), text="crushed aluminium can")
print("Plastic photo + 'crushed aluminium can':", answer["status"], "|", answer["message"])
show(assist(text="glass jar"))

**Result:** Warm-up took 32 seconds for 9 answers (about 3.6 s each); after that an image request takes about 0.12 to 0.15 s (mostly the CNN) and a description-only request about 0.01 s. No input and whitespace give "no input"; a missing file, a CSV passed as an image, gibberish, a Swahili phrase and a non-text input all give "need more information" with a reason; nothing crashed. Black (CNN 0.45), noise (0.54) and both real hold-out images (Plastic 0.65, Glass 0.54, both classified correctly) fall below 0.70 and ask for more, so two correct guesses are lost: that is the price of the cut-off. A usable description takes over, and the message says the photo was too unclear to use ("plastic bottle" gives Plastic; "crushed aluminium can" gives Metal). Trusting the description when the photo is unsure is reasonable (the photo is right only about 54% of the time there), but the text model has only been tested on template-style descriptions, and a real description with typos could be read wrongly with nothing to check it against.

## Step 12: End-to-End Test: 360 Requests

90 hold-out images (10 per category, random) in four situations: image only; with an agreeing description (a random Part 3 test description of the same category); with a conflicting description (one from another category); with gibberish.

In [ ]:
random_state = np.random.RandomState(0)
by_class = {c: [p for p in paths if os.path.basename(os.path.dirname(p)) == c] for c in CATEGORIES}
sample = [(c, p) for c in CATEGORIES for p in random_state.choice(by_class[c], 10, replace=False)]
description_pool = {c: test_data[test_data["category"] == c]["description"].tolist() for c in CATEGORIES}

rows = []
for true_category, image_path in sample:
    other = random_state.choice([c for c in CATEGORIES if c != true_category])
    agreeing = random_state.choice(description_pool[true_category])
    conflicting = random_state.choice(description_pool[other])
    for kind, text in [("image only", None), ("agreeing description", agreeing),
                       ("conflicting description", conflicting), ("gibberish description", "asdf qwerty")]:
        answer = assist(image_path=image_path, text=text)
        rows.append({"true": true_category, "kind": kind, "status": answer["status"],
                     "category": answer.get("category"), "seconds": answer["seconds"]})
test_results = pd.DataFrame(rows)

In [ ]:
rows = []
for kind, group in test_results.groupby("kind"):
    answered = group[group["status"].str.startswith("answered")]
    correct = None
    if kind != "conflicting description" and len(answered) > 0:
        correct = round(float((answered["category"] == answered["true"]).mean()), 3)
    rows.append({"kind": kind, "requests": len(group), "answered": len(answered),
                 "conflict": int((group["status"] == "conflict").sum()),
                 "need more info": int((group["status"] == "need more information").sum()),
                 "correct when answered": correct, "median seconds": round(float(group["seconds"].median()), 3)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** Image only: 74 of 90 answered, 16 sent back, 0.905 correct (about 7 wrong). Gibberish description: identical, so useless text changes nothing. Agreeing description: 83 answered, all correct, 7 conflicts: the 16 refused photos are answered from the description, and the 7 conflicts equal the 7 wrong image-only answers (the counts match; the images were not compared one by one), so every confident CNN error became a request to confirm. Conflicting description: 73 of 90 flagged (73 of the 74 with a usable photo); the 16 with an unusable photo are answered silently from the text, with a note. Median about 0.11 s. One random sample of 90; the descriptions are template-style, so this shows the combination logic, not accuracy on real user text.

## Step 13: Feedback

`give_feedback` records whether an answer helped and, optionally, the correct category (unknown request ids and categories are rejected). `learn_from_feedback` adds corrected descriptions to the training data and refits the text model, accepting the new model only if test accuracy does not fall. Image feedback is only logged, because retraining the CNN is too slow for a feedback loop.

In [ ]:
feedback_log = []

def give_feedback(request_id, helpful=None, correct_category=None, comment=""):
    if request_id not in request_log:
        return "unknown request id"
    if correct_category is not None:
        matches = [c for c in CATEGORIES if c.lower() == str(correct_category).strip().lower()]
        if not matches:
            return f"unknown category (choose from {CATEGORIES})"
        correct_category = matches[0]
    feedback_log.append({"request_id": request_id, "helpful": helpful, "correct_category": correct_category,
                         "comment": comment, "time": datetime.datetime.now().isoformat(timespec="seconds")})
    return "thank you, feedback recorded"

In [ ]:
def learn_from_feedback():
    global text_model
    corrections = []
    for entry in feedback_log:
        text = request_log[entry["request_id"]].get("input_text")
        if entry["correct_category"] and text:
            corrections.append((text, entry["correct_category"]))
    if not corrections:
        return "no usable corrections yet"

    new_text = pd.Series([clean_text(t) for t, c in corrections])
    new_y = pd.Series([label_to_id[c] for t, c in corrections])
    candidate = new_text_model()
    candidate.fit(pd.concat([dev_text, new_text], ignore_index=True), pd.concat([dev_y, new_y], ignore_index=True))

    old_accuracy = accuracy_score(test_data["y"], text_model.predict(test_data["clean"]))
    new_accuracy = accuracy_score(test_data["y"], candidate.predict(test_data["clean"]))
    if new_accuracy >= old_accuracy:
        text_model = candidate
        return f"accepted {len(corrections)} correction(s): test accuracy {old_accuracy:.4f} -> {new_accuracy:.4f}"
    return f"rejected: test accuracy would fall {old_accuracy:.4f} -> {new_accuracy:.4f}"

In [ ]:
before = assist(text="chupa ya plastiki")
print("before:", before["status"], "|", before["message"])
print(give_feedback(before["request_id"], helpful=False, correct_category="plastic", comment="plastic bottle in Swahili"))
print(learn_from_feedback())

after = assist(text="chupa ya plastiki")
print("after:", after["status"], "|", after["message"])
related = assist(text="chupa ya plastiki tupu")
print("related wording:", related["status"], "|", related["message"])
print("unrelated cases:", assist(text="red glass pen")["category"], "|", assist(text="banana peel")["category"])
print(pd.DataFrame(feedback_log))
print(give_feedback("nope"), "|", give_feedback(before["request_id"], correct_category="furniture"))

**Result:** Before: the Swahili phrase "chupa ya plastiki" (plastic bottle) was refused (no known words). After one correction to Plastic the guard accepted the refit (test accuracy 1.0000 to 1.0000) and the phrase, and a related wording sharing its words, are answered as Plastic, while "red glass pen" (Miscellaneous Trash) and "banana peel" (Food Organics) keep their answers. Invalid feedback (unknown id, unknown category) is rejected. The guard cannot catch a wrong correction: in a private check, labelling the same phrase Glass was also accepted (test accuracy stayed 1.0) and the model then answered Glass at about 0.79, because no test description contains those words. Corrections need human review or agreement from several users. The log lives in memory and is lost when the notebook closes.

## Summary

- **Design:** photo and/or description, then input checks, the CNN and the text classifier, a decision step (agree, conflict, unsure), then the generator with its checker and an answer cache. The answer comes with the policy text it used and a request id for feedback.
- **Evidence:** every invalid input gave a clean message; the CNN cut-offs come from measured accuracy by confidence (93% above 0.8, about 54% below 0.7); in 360 requests an agreeing description turned all seven confident CNN errors into requests to confirm; a request takes about 0.11 s after a 32 s warm-up; one correction made a refused phrase answerable without harming other answers.
- **Limitations:** the CNN cannot recognise non-waste pictures (only synthetic ones tested); the text classifier and generator are tested only on template-style data; when a photo is unsure a description is trusted unchecked; the feedback guard cannot detect a wrong correction and the log is not saved; held-out images are partly the validation set; each experiment is a single run on a small sample.